# 용접에 대한 이해

용접(Welding)은 금속·비금속 재료에 열·압력(또는 둘 다)을 가해 원자 간 결합을 유도해 접합하는 공정입니다. 전통적으로 열역학·물리학 기반 공정이었으나, 최근에는 비전 센서와 딥러닝/강화학습을 결합한 지능형 로봇 용접 시스템으로 빠르게 진화하고 있습니다. 이 절은 본 프로젝트(저항 점용접의 적응형 열입력 보정)를 이해하는 데 필요한 배경을 간단히 정리합니다.

## 1. 주요 용접 종류

에너지 전달·보호 방식에 따라 다음과 같이 분류됩니다.

| 분류 | 공정 | 특징 |
|---|---|---|
| 아크 용접 | GMAW(Gas Metal Arc Welding)/MIG(Metal Inert Gas)·MAG(Metal Active Gas) | 와이어 연속 송급 + 보호가스. 생산성이 높아 로봇 자동화의 80% 이상을 차지 |
| | GTAW(Gas Tungsten Arc Welding)/TIG(Tungsten Inert Gas) | 비소모성 텅스텐 전극, 정밀·고품질이나 속도가 느림 (배관·항공) |
| | SMAW(Shielded Metal Arc Welding, 피복아크) | 수동 용접의 기본형. 야외·보수 시공용 |
| 고에너지 빔 용접 | 레이저 | 높은 에너지 밀도, 좁은 열영향부(HAZ, Heat Affected Zone), 고속 접합 (이차전지·정밀 전자) |
| | 전자빔(EBW, Electron Beam Welding) | 진공 중 고전압 전자빔으로 깊은 용입 형성 (원자력·우주항공) |
| **저항 용접** | **스폿/심 (본 프로젝트 대상)**⭐ | **접촉저항열($Q=I^2Rt$)과 가압을 이용한 무가스 접합 (자동차 차체 조립)** |
| 고상 용접 | 마찰교반(FSW, Friction Stir Welding) | 모재를 녹이지 않고 소성 유동으로 접합 (알루미늄·이종 금속) |

## 2. 핵심 물리 변수와 상관관계

용접 품질(용입 깊이, 비드 형상, 기공·균열, 열변형)은 여러 물리 변수의 비선형 상호작용으로 결정됩니다.

- **주요 변수**: 전류 $I$, 전압 $V$, 속도 $v$, 송급 속도(WFS, Wire Feed Speed), 토치 각도, 루트 갭·베벨 각도·단차, 모재 두께·곡률, 표면 청정도, 보호가스 조성, 예열 온도 등
- **입열량(Heat Input)**: $H=\eta \cdot \dfrac{V \cdot I}{v}$ ($\eta$: 열효율). $V\cdot I$가 높거나 $v$가 느릴수록 입열량이 늘어 용입은 깊어지지만, 결정립 조대화로 인성이 떨어지고 열변형(잔류응력)이 커집니다. 본 프로젝트에서 다루는 저항용접 입열량 $Q=I^2Rt$도 같은 관계의 변형입니다.
- **전류 vs 전압**: 전류는 용입 깊이·용융 속도를, 전압은 아크 길이·비드 폭·젖음성을 지배합니다.
- **갭·굴곡의 영향**: 루트 갭이 넓어지거나 모재가 휘어(곡률) 있으면 열전달과 접촉 상태가 표준 조건에서 벗어나 결함(번스루, 언더필, 용융풀 쏠림) 위험이 커집니다. **이 노트북이 다루는 "표면온도·굴곡 편차"가 바로 이 문제입니다.**⭐
- **습도**: 수분이 아크 플라즈마에서 해리되어 확산성 수소로 유입되면 지연균열(Cold Cracking)·기공을 유발합니다.

## 3. AI 기반 로봇 자동 대응 용접 시스템

기존 룰 기반 티칭(Teaching & Playback) 로봇은 부품 공차, 열변형에 의한 갭 변동, 모재 편차에 대응하지 못해 불량이 발생합니다. 최신 AI 용접 시스템은 다음 파이프라인으로 이를 보완합니다.

```
[센서] 비전 프로파일러 · 열화상 · 전류/전압 파형
   ↓
[AI 추론] CNN(Convolutional Neural Network)/ViT(Vision Transformer)(결함·용융풀 탐지) · LSTM(Long Short-Term Memory, 파형 기반 스패터 예측) · Physics-Informed ML(물리 정보 결합 머신러닝)/DRL(Deep Reinforcement Learning, 파라미터 최적화)
   ↓
[액추에이터] 로봇 궤적 보정(Adaptive Seam Tracking) + 용접기 파라미터 실시간 동기화
```

즉 센서로 실시간 편차를 감지하고, 학습된 모델로 전류·전압·속도 등 공정 변수를 즉시 재계산해 로봇과 용접기에 반영하는 구조입니다.

**본 프로젝트와의 연결:** 이 노트북(RSW(Resistance Spot Welding, 저항 점용접) 적응형 파인튜닝)은 위 파이프라인 중 "표면온도·굴곡 편차를 감지해 저항 점용접의 열입력 예측을 실시간 보정"하는 부분에 해당합니다. 즉 고정된 표준값(baseline)만 적용하는 시스템과, 편차를 감지해 보정하는 적응형(adaptive) 시스템을 비교해 후자의 효용을 정량적으로 검증하는 것이 이 노트북의 목표입니다.

## RSW(Resistance Spot Welding, 저항 점용접) 실시간 적응형 파인튜닝 — 굴곡·표면온도 편차 대응 시뮬레이션

**한눈에 보는 구조.** 이 노트북은 **실제로 있었던 용접 493건**을 뼈대로 삼되, 그 위에 "이 용접 당시 표면온도·굴곡이 이만큼 벗어나 있었다면"이라는 **가상의 편차 시나리오를 인위적으로 주입**합니다. 그리고 그 편차를 전혀 모르는 baseline과, 센서로 감지·보정하는 adaptive 중 어느 쪽이 더 나은 판정을 내리는지 비교합니다. 즉 데이터 전체가 가짜인 것이 아니라 **실측 위에 특정 항목만 가상으로 덧씌운 하이브리드 구조**이며, 어떤 컬럼이 실측이고 어떤 컬럼이 합성(SIMULATED)인지는 A-3에서 표로 정리합니다.

### A-1 — 데이터 출처

이 노트북이 사용하는 실측 데이터는 `result_rsw/step1_aggregated_samples.csv`이며, 새로 수집한
자료가 아니라 `1_rsw_optimization.ipynb`의 B-1c(`ResistanceWeldingOptimizer.step1_load_and_aggregate`)가
원본 데이터셋을 집계해 만든 산출물을 그대로 재사용한 것입니다. 원본 데이터셋의 출처는 다음과 같습니다.

- **원자료:** Resistance Spot Welding Insights (Mendeley Data, CC BY 4.0)
- **원본 규모:** 용접 495건에 대한 통전 중 계측치(전류·통전시간·가압력·전극각도·판재두께·너겟지름·
  인장강도·결함 라벨)
- **정제 결과:** 전류 계측 오류(결측 또는 센서 오류값 -99)가 있는 행을 제외해 **493건**을 샘플
  단위로 집계함(가압력 {35, 60, 80, 95} psi, 통전시간 0.2–1.5초 7단계, 전극각도 {0, 15}도)
- **본 노트북에서의 위치:** 이 493건은 B-2에서 7:2:1로 분할되어, 표준값 학습(train)과 성능 비교
  (valid/test)의 실측 기반으로 사용됩니다.

### A-2 — 용어 정의

본 노트북에서 반복적으로 등장하는 용어를 이론 전개에 앞서 정의합니다.

| 용어 | 기호/컬럼명 | 물리적 의미 |
|---|---|---|
| 전류 | `avg_current_A` | 통전 구간 동안 측정된 평균 전류(A). 값이 클수록 단위 시간당 발열량이 증가함 |
| 통전시간 | `weld_time_s` | 전극 사이에 전류가 흐른 시간(초). 열입력은 전류의 제곱과 이 시간에 비례함 |
| 가압력 | `pressure_psi` | 전극이 모재를 누르는 압력(psi). 접촉저항과 전류 밀도 분포에 영향을 줌 |
| 전극각도 | `angle_deg` | 전극과 모재 표면이 이루는 기울기(도). 0도는 수직 접촉, 15도는 기울어진 접촉을 의미함 |
| 열입력 proxy | $Q \propto I^2 t$ | 접촉저항 $R$을 상수로 가정한 상대적 발열량 지표. 절대 물리량이 아니라 조건 간 비교를 위한 지표임 |
| 접촉저항 | $R$ | 전극-모재 접촉면의 전기저항. 원 데이터셋에 계측되어 있지 않아 $R=1$로 정규화되어 있음 |
| 너겟 지름 | `nugget_diameter_mm` | 용접부에 형성된 용융 접합부(너겟)의 지름(mm). 접합 강도와 직결됨 |
| 인장강도 | `pull_test_N` | 용접부를 잡아당겨 파단시키는 데 필요한 힘(N). 접합 품질의 최종 검증 지표 |
| 미융착(Bad) | `is_bad`, `category=="Bad"` | 열입력 부족으로 모재가 충분히 융착되지 않은 결함 |
| 팽출(Explode) | `is_explode`, `category=="Explode"` | 과도한 열 또는 기울어진 접촉으로 금속이 튀어나가는 결함 |
| 굴곡 편차 | `sim_curvature_dev_deg` | (합성) 용접 대상 표면의 곡률로 인해 실제 접촉각이 `angle_deg`에서 추가로 벗어나는 정도 |
| 표면온도 편차 | `sim_temp_dev_C` | (합성) 표준(가정) 조건 대비 용접 대상 표면온도가 벗어난 정도(절대 온도가 아닌 상대값, A-6 참고). 접촉저항 $R$을 변화시키는 요인으로 사용함 |
| RSW | Resistance Spot Welding | 이 노트북이 다루는 용접 공정(저항 점용접). 전극 사이에 전류를 흘려 접촉저항 발열로 두 금속판을 국소적으로 녹여 붙이는 방식 |
| SGD | Stochastic Gradient Descent | 데이터를 무작위 미니배치로 나눠 순차적으로 파라미터를 갱신하는 경사하강법. B-7 센서 보정 학습에 사용 |
| MSE | Mean Squared Error | 예측값과 참값의 오차를 제곱해 평균한 손실함수. D-5 학습곡선의 Loss 단위 |
| SEM | Standard Error of the Mean | 표본평균의 표준오차(표준편차/√n). 구간별 실측 평균±SEM 에러바에 사용 |
| TP/FP/FN/TN | True/False Positive/Negative | 혼동행렬의 4가지 판정 결과(맞은 양성/틀린 양성/틀린 음성/맞은 음성). D-2에서 사용 |

### A-3 — 실측값 vs 합성(SIMULATED)값 구분

이 노트북은 실측 데이터와 합성 데이터를 함께 다루므로, 본격적인 분석에 앞서 어떤 변수가 어느
쪽에 속하는지 명확히 구분합니다. **이하 모든 절에서 `sim_`, `obs_`로 시작하는 컬럼과 그로부터
파생된 `_true`, `_adapt` 접미사가 붙은 컬럼은 전부 합성값입니다.**⭐

| 변수 | 성격 | 근거 |
|---|---|---|
| `avg_current_A`, `weld_time_s`, `pressure_psi`, `angle_deg` | 실측 | 원본 계측 데이터 기반 집계(A-1) |
| `heat_input_proxy`, `category`, `is_bad`, `is_explode`, `nugget_diameter_mm`, `pull_test_N` | 실측 원값 또는 그 유도값 | 원본 계측치로부터 직접 기록되거나 계산된 값 |
| `sim_temp_dev_C`, `sim_curvature_dev_deg` | 합성(SIMULATED) | 물리적으로 동기부여되었으나 실측이 아님(B-5) |
| `obs_temp_dev_C`, `obs_curvature_dev_deg` | 합성(SIMULATED) | 위 합성값에 가정된 센서 관측 노이즈를 더한 값(B-6) |
| `Q_effective_true`, `effective_tilt_true_deg`, `true_bad_risk`, `true_explode_risk` | 합성(SIMULATED) | 합성 편차를 표준값 모델에 대입한 가상 시나리오(B-8). 실제 관측 결과가 아님 |
| `Q_adapt`, `tilt_adapt`, `adaptive_bad_risk`, `adaptive_explode_risk` | 합성(SIMULATED) | 보정된 센서 관측값 기반 예측(B-7, B-8) |

이 구분은 F-2에서 결과 해석의 관점에서 다시 상세히 다룹니다.

### A-4 — 왜 이 노트북이 필요한가

`1_rsw_optimization.ipynb`는 저항 점용접(RSW) 실측 493건에 물리 법칙 기반 곡선을 피팅해 표준 조건에서
안전한 열입력 하한과 위험한 전극각도를 도출했습니다. 그러나 실제 생산 현장에서는 모든 용접이 동일한
표준 조건에서 이뤄지지 않습니다. 용접 대상 판재가 미세하게 휘어 있거나(굴곡), 앞선 공정의 열이 남아
표면 온도가 예열 상태이거나 주변 환경에 따라 다르면, 동일한 전류·통전시간·가압력을 인가해도 실제로
전달되는 열과 접촉 상태는 표준 조건과 달라집니다.

이 노트북은 표준값을 고정 적용하는 시스템과, 굴곡·표면온도 편차를 실시간으로 감지해 대응하는
적응형 시스템을 비교하여, 후자가 실제로 더 나은 판정을 내리는지 정량적으로 검증합니다.

**핵심 전제를 먼저 밝힙니다.** 전류·통전시간·가압력·전극각도는 493건 모두 실측값입니다. 그러나
굴곡과 표면온도 편차, 그리고 이를 감지하는 센서 관측값은 이 노트북에서 물리적으로 타당하게
합성(SIMULATED)한 것으로, 실제로 측정되거나 관측된 값이 아닙니다(A-3 표 참고). 이런 조합을 만족하는
공개 데이터셋이 존재하지 않아(조사 경위는 F-1 참고), 실측 기반과 합성 기반을 명확히 구분해 결합하는
하이브리드 방식을 채택했습니다.

### A-5 — 무엇을, 어떻게 검증할 것인가

핵심 질문은 두 가지입니다.⭐

1. 표면온도가 표준(가정) 조건에서 벗어나면 열입력이 달라져 미융착(Bad) 위험 판정이 바뀌는가?
2. 판재 굴곡으로 인한 추가 접촉각 편차가 팽출(Explode) 위험 판정을 바꾸는가?

각 질문에 대해 세 가지 예측 방식을 비교합니다.

| 예측 방식 | 편차를 아는가 | 설명 |
|---|---|---|
| 참값(ground truth) | 완전히 앎(시뮬레이션 내부값) | 이런 조건이 실제로 있었다면 어떤 결과였을지에 대한 물리모델 기반 가상 시나리오 — 실측 결과가 아님 |
| baseline | 전혀 모름 | 표준값(R=1, 편차 없음)을 고정 적용 — 현재 대다수 현장 관행에 해당 |
| adaptive | 노이즈가 섞인 센서로 추정 | 온도·굴곡 센서(가정)로 편차를 감지하고, 별도 검증 세트로 보정한 값을 반영 |

### A-6 — 물리적 근거

$$Q = I^2 R t$$

`1_rsw_optimization.ipynb`는 접촉저항 $R$을 계측하지 못해 $R=1$로 정규화했습니다. 그러나 금속의
전기저항은 온도가 오르면 함께 증가한다는 것이 잘 알려진 물리 현상입니다. 이 노트북은 표면온도
편차가 $R$을 선형적으로 변화시킨다고 가정한 근사 모델 $R_{eff} = 1 + \alpha \Delta T$
($\alpha=0.004/°C$, 철강 계열의 전형적 근사값이며 이 데이터셋에서 측정된 값은 아님⭐)을 사용해
$Q_{eff} = I^2 R_{eff} t$를 계산합니다.

**$\Delta T$는 절대 온도가 아니라 상대값입니다.** "표준(가정) 조건"에 구체적인 섭씨 값(예: 25°C)이 정해져 있는 것이 아니라, B-3에서 $Q_{min}$·$k$를 피팅한 **TRAIN 345건이 용접될 당시의(측정되지 않은, 암묵적) 온도 상태**를 기준점(ΔT=0)으로 삼습니다. `sim_temp_dev_C`는 그 기준점 대비 표면온도가 몇 도 더 높거나 낮은지를 나타낼 뿐이며, $R_{eff}=1+\alpha\Delta T$ 공식이 곱셈 형태라 절대 온도값 자체는 필요하지 않습니다.

굴곡은 전극-모재 접촉각을 실측 `angle_deg`에 추가로 편향시키는 요인으로 모델링합니다
($\text{effective\_tilt} = \text{angle\_deg} + \Delta\theta_{curvature}$). `1_rsw_optimization.ipynb`의
B-5는 `angle_deg`가 0°/15° 두 수준뿐인 범주형이라 연속 로지스틱을 피팅할 수 없었다는 한계를 남겼는데,
이 노트북의 합성 굴곡 변수는 처음으로 연속적인 접촉각 편차를 다루되, 이는 실측의 확장이 아니라
방법론 시연을 위한 합성적 확장이라는 점을 다시 강조합니다.

### A-7 — 이 노트북을 읽는 방법

B 섹션에서 표준값 학습, 합성 편차·센서 생성, 보정, 예측기 3종을 순서대로 구현합니다. C 섹션은
분할과 합성 변수 분포를 확인합니다. D 섹션은 baseline과 adaptive의 성능 지표(정확도·정밀도·재현율·F1)를
Bad·Explode 각 과제별로 비교하고, 실시간 재생 애니메이션으로 시연합니다. E 섹션은 결과를 종합
해석하며, F 섹션은 이 방법론의 한계와 정직성 표기를 정리합니다.⭐ **F-2를 반드시 읽어주십시오** — 이
노트북의 결과를 어떻게 해석해야 하는지에 대한 가장 중요한 안내가 담겨 있습니다.


## B. 문제와 해결 및 구현

### B-1 — 라이브러리 로드 및 데이터 준비

A-1에서 밝힌 대로, `1_rsw_optimization.ipynb`가 이미 검증해 저장한 샘플 단위 집계 데이터
(`result_rsw/step1_aggregated_samples.csv`, 493건)를 그대로 재사용합니다. 원본 CSV를 다시 파싱하지
않는 이유는, 그 전처리(전류 계측 오류 행 제외, 샘플 단위 집계, 열입력 proxy 계산)가 이미 별도
노트북에서 검증되었기 때문입니다.

In [ ]:
# ==========================================
# [B-1] 라이브러리 로드 및 데이터 준비
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트 -- 없으면 그래프의 한글이 깨짐(1_rsw_optimization.ipynb와 동일)
plt.rcParams['axes.unicode_minus'] = False     # 한글 폰트 사용 시 마이너스 기호 깨짐 방지
plt.rcParams.update({
    "axes.labelsize": 13,
    "axes.labelweight": "bold",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "font.size": 11,
})

RESULT_DIR = "result_rsw_adaptive"
FIGURE_DIR = os.path.join(RESULT_DIR, "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

AGG_CSV_PATH = os.path.join("result_rsw", "step1_aggregated_samples.csv")
agg = pd.read_csv(AGG_CSV_PATH)
print(f"[안내] 실측 집계 데이터 로드: {len(agg)}건 (1_rsw_optimization.ipynb의 step1 산출물 재사용)")
print(f"[안내] category 분포:\n{agg['category'].value_counts().to_string()}")
agg.head()


### B-2 — 7:2:1 층화 분할

`category`(Good/Bad/Explode) 비율을 유지하는 층화(stratified) 분할을 적용합니다. Bad·Explode
표본이 원래도 희소하므로(각각 4.3%, 6.3%), 층화하지 않으면 test에 결함 표본이 거의 남지 않아
precision·recall·F1이 정의되지 않거나 무의미해질 위험이 있습니다. `random_state`를 고정해 재현
가능하게 합니다.

In [ ]:
# ==========================================
# [B-2] 7:2:1 층화 분할 (train:valid:test)
# ==========================================
RANDOM_STATE = 42

train_df, temp_df = train_test_split(
    agg, test_size=0.3, stratify=agg["category"], random_state=RANDOM_STATE)
valid_df, test_df = train_test_split(
    temp_df, test_size=(1.0 / 3.0), stratify=temp_df["category"], random_state=RANDOM_STATE)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame([
    {"split": "train", "n": len(train_df), "n_bad": int(train_df["is_bad"].sum()), "n_explode": int(train_df["is_explode"].sum())},
    {"split": "valid", "n": len(valid_df), "n_bad": int(valid_df["is_bad"].sum()), "n_explode": int(valid_df["is_explode"].sum())},
    {"split": "test", "n": len(test_df), "n_bad": int(test_df["is_bad"].sum()), "n_explode": int(test_df["is_explode"].sum())},
])
split_summary.to_csv(os.path.join(RESULT_DIR, "step1_split_summary.csv"), index=False)
print(f"[분할] train={len(train_df)}건, valid={len(valid_df)}건, test={len(test_df)}건 "
      f"(전체 {len(agg)}건, category 층화, random_state={RANDOM_STATE})")
split_summary


### B-3 — 표준값(train) Bad 물리모델 피팅

`1_rsw_optimization.ipynb`의 `ResistanceWeldingOptimizer.bad_rate_logistic_model`과 그 피팅 절차를
그대로 재사용합니다(구간화하지 않고 원본 샘플 단위로 피팅하며, bounds로 비물리적 해를 차단하는
방식은 그 노트북에서 이미 검증됨). 단, 여기서는 493건 전체가 아니라 **train 345건만**으로 피팅해
표준값을 정의합니다.

In [ ]:
# ==========================================
# [B-3] 표준값(train) Bad(열입력 부족) 물리모델 피팅
# ==========================================
def bad_rate_logistic_model(Q, Q_min, k):
    """열입력이 Q_min 아래로 떨어질수록 미융착(Bad) 확률이 증가하는 감소형 시그모이드.
    1_rsw_optimization.ipynb의 ResistanceWeldingOptimizer.bad_rate_logistic_model과 동일한 함수 및
    피팅 절차를 재사용한다(원본 샘플 단위 피팅 + bounds로 비물리적 해 차단이 이미 검증된 방식)."""
    return 1.0 / (1.0 + np.exp(k * (Q - Q_min)))

x_train = train_df["heat_input_proxy"].values
y_train = train_df["is_bad"].values.astype(float)
x_range_train = x_train.max() - x_train.min()

p0 = [np.percentile(x_train, 15), 10.0 / x_range_train]
bounds = ([x_train.min(), 1e-8], [x_train.max(), 50.0 / x_range_train])

popt, pcov = curve_fit(bad_rate_logistic_model, x_train, y_train, p0=p0, bounds=bounds, maxfev=20000)
Q_min_train, k_train = popt
Q_min_err_train = np.sqrt(np.diag(pcov))[0]
reliability_train = Q_min_err_train / Q_min_train

bad_fit_summary = pd.DataFrame([{
    "Q_min_train": Q_min_train, "Q_min_err_train": Q_min_err_train,
    "k_train": k_train, "reliability_ratio": reliability_train,
    "n_train": len(train_df),
}])
bad_fit_summary.to_csv(os.path.join(RESULT_DIR, "step2_bad_fit_train.csv"), index=False)
verdict = "신뢰 가능" if reliability_train < 0.3 else "주의(오차비율 높음)"
print(f"[표준값] Q_min_train = {Q_min_train:.3e} ± {Q_min_err_train:.3e} "
      f"(오차비율 {reliability_train * 100:.1f}%, {verdict})")
bad_fit_summary


### B-4 — 표준값(train) Explode 임계 규칙

`angle_deg`는 0도/15도 두 수준뿐인 범주형이라 (`1_rsw_optimization.ipynb`의 B-5에서 이미 확인) 연속
로지스틱을 피팅할 수 없습니다. 표준 규칙은 데이터로 피팅한 값이 아니라, 실측 두 수준(0도는 안전,
15도는 위험 증가) 사이의 단순 중간점을 임계값으로 고정하는 규칙 기반 판정입니다.

In [ ]:
# ==========================================
# [B-4] 표준값(train) Explode(팽출) 임계 규칙
# ==========================================
# angle_deg는 0도/15도 두 수준뿐인 범주형이라 연속 로지스틱을 피팅할 수 없다
# (1_rsw_optimization.ipynb의 B-5에서 이미 확인됨). 표준 규칙은 두 실측 수준의 중간값을 단순
# 임계값으로 고정한다 -- 데이터로 피팅한 값이 아니라 실측 두 수준 사이의 단순 중간점임을 명시한다.
EXPLODE_TILT_THRESHOLD_DEG = 7.5

train_angle_rates = train_df.groupby("angle_deg")["is_explode"].mean()
print("[표준값] train 내 전극각도별 Explode 비율:")
print(train_angle_rates.to_string())
print(f"[표준값] Explode 위험 판정 임계값(고정 규칙) = {EXPLODE_TILT_THRESHOLD_DEG}도")


### B-5 — 합성 굴곡·표면온도 편차 생성 (SIMULATED)

train은 편차 0으로 고정합니다. B-3·B-4의 표준값 학습이 순수 실측 조건을 기준으로 이뤄졌다는
전제와 일관성을 유지하기 위함입니다. valid+test에만 표준 조건을 벗어난 가상 시공 상황을 합성
주입합니다. 편차의 분포 범위(균등분포 [-15,45], [0,20])와 온도계수 $\alpha$는 모두 임의로 정한
가정값이며, 이 데이터셋에서 측정되거나 문헌에서 인용한 값이 아닙니다(A-3 참고).

In [ ]:
# ==========================================
# [B-5] 합성(SIMULATED) 굴곡·표면온도 편차 생성
# ==========================================
ALPHA_R_PER_C = 0.004  # 철강 계열 전형적 저항 온도계수 근사값 -- 이 데이터셋에서 측정된 값이 아님

rng = np.random.default_rng(RANDOM_STATE)

for df_part in (valid_df, test_df):
    n = len(df_part)
    df_part["sim_temp_dev_C"] = rng.uniform(-15, 45, size=n)
    df_part["sim_curvature_dev_deg"] = rng.uniform(0, 20, size=n)

train_df["sim_temp_dev_C"] = 0.0
train_df["sim_curvature_dev_deg"] = 0.0

for df_part in (train_df, valid_df, test_df):
    df_part["R_eff_true"] = 1.0 + ALPHA_R_PER_C * df_part["sim_temp_dev_C"]
    df_part["Q_effective_true"] = (df_part["avg_current_A"] ** 2) * df_part["R_eff_true"] * df_part["weld_time_s"]
    df_part["effective_tilt_true_deg"] = df_part["angle_deg"] + df_part["sim_curvature_dev_deg"]

sim_summary = pd.concat([train_df.assign(split="train"), valid_df.assign(split="valid"),
                          test_df.assign(split="test")])[
    ["split", "sample_id", "sim_temp_dev_C", "sim_curvature_dev_deg",
     "Q_effective_true", "effective_tilt_true_deg"]]
sim_summary.to_csv(os.path.join(RESULT_DIR, "step3_synthetic_deviations.csv"), index=False)
print(f"[합성 SIMULATED] valid+test {len(valid_df) + len(test_df)}건에 굴곡·표면온도 편차 주입 완료 "
      f"(train {len(train_df)}건은 편차 0으로 고정 -- 실측 조건 그대로)")
sim_summary.groupby("split")[["sim_temp_dev_C", "sim_curvature_dev_deg"]].describe().T


### B-6 — 센서 관측 모델 (노이즈 포함, SIMULATED)

실시간 감지는 완벽하지 않습니다. 접촉식 또는 적외선 온도센서와 비전 기반 곡률 추정에 각각 합리적인
수준의 관측 노이즈가 있다고 가정합니다.⭐ 이 노이즈 크기 역시 실측이 아니라 가정값입니다.

In [ ]:
# ==========================================
# [B-6] 합성(SIMULATED) 센서 관측 모델 -- 노이즈 포함
# ==========================================
for df_part in (valid_df, test_df):
    n = len(df_part)
    df_part["obs_temp_dev_C"] = df_part["sim_temp_dev_C"] + rng.normal(0, 3.0, size=n)
    df_part["obs_curvature_dev_deg"] = df_part["sim_curvature_dev_deg"] + rng.normal(0, 1.5, size=n)

print("[센서 SIMULATED] valid+test에 노이즈 섞인 관측값(obs_temp_dev_C, obs_curvature_dev_deg) 생성 완료")
valid_df[["sim_temp_dev_C", "obs_temp_dev_C", "sim_curvature_dev_deg", "obs_curvature_dev_deg"]].head()


### B-7 — VALID 세트 기반 센서 보정(calibration) -- 경사하강법 Epoch 학습

TEST로 직접 보정하면 데이터 유출(leakage)이 되므로, 반드시 VALID만 사용해 보정계수를 구한 뒤
TEST에 적용합니다. 이 노트북의 핵심 주제인 "실시간 파인튜닝"을 문자 그대로 구현하기 위해, 닫힌해
(closed-form) 최소자승 대신 **경사하강법으로 여러 epoch에 걸쳐 점진적으로 보정계수를 학습**합니다.
VALID(약 98건)를 다시 calib_train/calib_valid로 나눠, calib_train으로 파라미터를 갱신하고
calib_valid로 매 epoch 성능을 추적합니다 -- 이 과정에서도 TEST는 전혀 사용하지 않습니다. 학습
이력 시각화는 D-5를 참고하십시오. calib_train/calib_valid 분할은 `category`(Good/Bad/Explode)로
층화해, valid_df에 원래 적은 Bad·Explode 표본이 한쪽에 쏠리지 않도록 했습니다. 매 epoch
calib_train을 무작위로 섞어 배치 크기 16의 미니배치로 나눠 SGD로 갱신합니다(전체 배치
경사하강법과 달리 배치 구성이 매번 달라져, 실제 파인튜닝처럼 epoch별 loss에 자연스러운
요동이 생깁니다). 배치별 학습률은 원래 학습률을 그 epoch의 배치 수로 나눈 값을 써서,
한 epoch 안에서 이뤄지는 전체 업데이트 크기가 원래의 전체배치 경사하강법과 비슷하게 유지되도록 했습니다 -- 이 조정이 없으면 배치 수만큼(약 5배) 학습률이 커진 것과 같아져 손실이 완만히 요동치는 대신 극단적으로 튀는 문제가 있었습니다.

In [ ]:
# ==========================================
# [B-7] VALID 세트 기반 센서 보정 -- 경사하강법 반복 파인튜닝 (Epoch 학습)
# ==========================================
calib_train_df, calib_valid_df = train_test_split(
    valid_df, test_size=0.3, stratify=valid_df["category"], random_state=RANDOM_STATE)
calib_train_df = calib_train_df.reset_index(drop=True)
calib_valid_df = calib_valid_df.reset_index(drop=True)

calib_valid_df["true_bad_risk"] = (bad_rate_logistic_model(
    calib_valid_df["Q_effective_true"], Q_min_train, k_train) > 0.5).astype(int)
calib_valid_df["true_explode_risk"] = (
    calib_valid_df["effective_tilt_true_deg"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)


def standardize(x):
    mu, sigma = x.mean(), x.std()
    sigma = sigma if sigma > 1e-8 else 1.0
    return mu, sigma


mu_obs_temp, sigma_obs_temp = standardize(calib_train_df["obs_temp_dev_C"].values)
mu_obs_curv, sigma_obs_curv = standardize(calib_train_df["obs_curvature_dev_deg"].values)

z_obs_temp_train = (calib_train_df["obs_temp_dev_C"].values - mu_obs_temp) / sigma_obs_temp
z_obs_curv_train = (calib_train_df["obs_curvature_dev_deg"].values - mu_obs_curv) / sigma_obs_curv
true_temp_train = calib_train_df["sim_temp_dev_C"].values
true_curv_train = calib_train_df["sim_curvature_dev_deg"].values

z_obs_temp_valid = (calib_valid_df["obs_temp_dev_C"].values - mu_obs_temp) / sigma_obs_temp
z_obs_curv_valid = (calib_valid_df["obs_curvature_dev_deg"].values - mu_obs_curv) / sigma_obs_curv
true_temp_valid = calib_valid_df["sim_temp_dev_C"].values
true_curv_valid = calib_valid_df["sim_curvature_dev_deg"].values

# 초기값: a=0(아직 센서를 신뢰하지 않음), b=calib_train 평균(정보가 없을 때의 기본 예측) --
# "학습이 진행되며 점차 센서를 신뢰하게 된다"는 실시간 파인튜닝의 취지를 초기화에도 반영한다
a_temp_z, b_temp_z = 0.0, float(true_temp_train.mean())
a_curv_z, b_curv_z = 0.0, float(true_curv_train.mean())

N_EPOCHS = 60
LR0 = 0.08
LR_DECAY = 0.97

# 실제 파인튜닝처럼 매 epoch 미니배치로 나눠 SGD 학습한다(전체 배치 경사하강법과 달리
# 배치 구성이 매 epoch 달라져 loss 곡선에 자연스러운 요동이 생긴다). 배치별 학습률은
# lr/배치수로 나눠, 한 epoch 안에서 이뤄지는 전체 업데이트 크기가 원래의 전체배치 경사하강법과
# 비슷하게 유지되도록 한다 -- 그래야 미세한 요동만 생기고 극단적으로 발산하지 않는다.
BATCH_SIZE_D4 = 16
batch_rng_d4 = np.random.default_rng(RANDOM_STATE)
n_calib_train_d4 = len(z_obs_temp_train)
n_batches_d4 = int(np.ceil(n_calib_train_d4 / BATCH_SIZE_D4))

epoch_history = []
for epoch in range(1, N_EPOCHS + 1):
    lr = LR0 * (LR_DECAY ** (epoch - 1))
    batch_order_d4 = batch_rng_d4.permutation(n_calib_train_d4)
    lr_batch = lr / n_batches_d4
    for batch_start in range(0, n_calib_train_d4, BATCH_SIZE_D4):
        idx = batch_order_d4[batch_start:batch_start + BATCH_SIZE_D4]
        z_obs_temp_batch = z_obs_temp_train[idx]
        z_obs_curv_batch = z_obs_curv_train[idx]
        true_temp_batch = true_temp_train[idx]
        true_curv_batch = true_curv_train[idx]

        pred_temp_batch = a_temp_z * z_obs_temp_batch + b_temp_z
        resid_temp_batch = pred_temp_batch - true_temp_batch
        grad_a_temp = 2.0 * np.mean(resid_temp_batch * z_obs_temp_batch)
        grad_b_temp = 2.0 * np.mean(resid_temp_batch)

        pred_curv_batch = a_curv_z * z_obs_curv_batch + b_curv_z
        resid_curv_batch = pred_curv_batch - true_curv_batch
        grad_a_curv = 2.0 * np.mean(resid_curv_batch * z_obs_curv_batch)
        grad_b_curv = 2.0 * np.mean(resid_curv_batch)

        a_temp_z -= lr_batch * grad_a_temp
        b_temp_z -= lr_batch * grad_b_temp
        a_curv_z -= lr_batch * grad_a_curv
        b_curv_z -= lr_batch * grad_b_curv

    train_loss_temp = float(np.mean((a_temp_z * z_obs_temp_train + b_temp_z - true_temp_train) ** 2))
    train_loss_curv = float(np.mean((a_curv_z * z_obs_curv_train + b_curv_z - true_curv_train) ** 2))

    pred_temp_valid = a_temp_z * z_obs_temp_valid + b_temp_z
    pred_curv_valid = a_curv_z * z_obs_curv_valid + b_curv_z
    valid_loss_temp = float(np.mean((pred_temp_valid - true_temp_valid) ** 2))
    valid_loss_curv = float(np.mean((pred_curv_valid - true_curv_valid) ** 2))

    # 표준화 공간의 (a_z, b_z)를 원래 스케일의 corrected = a*obs + b 형태로 환산
    a_temp_epoch = a_temp_z / sigma_obs_temp
    b_temp_epoch = b_temp_z - a_temp_z * mu_obs_temp / sigma_obs_temp
    a_curv_epoch = a_curv_z / sigma_obs_curv
    b_curv_epoch = b_curv_z - a_curv_z * mu_obs_curv / sigma_obs_curv

    corrected_temp_valid = a_temp_epoch * calib_valid_df["obs_temp_dev_C"] + b_temp_epoch
    corrected_curv_valid = a_curv_epoch * calib_valid_df["obs_curvature_dev_deg"] + b_curv_epoch
    Q_adapt_valid = (
        (calib_valid_df["avg_current_A"] ** 2)
        * (1.0 + ALPHA_R_PER_C * corrected_temp_valid)
        * calib_valid_df["weld_time_s"]
    )
    tilt_adapt_valid = calib_valid_df["angle_deg"] + corrected_curv_valid

    pred_bad_valid = (bad_rate_logistic_model(Q_adapt_valid, Q_min_train, k_train) > 0.5).astype(int)
    pred_explode_valid = (tilt_adapt_valid > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

    combined_true = ((calib_valid_df["true_bad_risk"] == 1) | (calib_valid_df["true_explode_risk"] == 1)).astype(int)
    combined_pred = ((pred_bad_valid == 1) | (pred_explode_valid == 1)).astype(int)
    epoch_f1 = f1_score(combined_true, combined_pred, zero_division=0)
    epoch_acc = accuracy_score(combined_true, combined_pred)

    epoch_history.append({
        "epoch": epoch, "lr": lr,
        "train_loss_temp": train_loss_temp, "train_loss_curv": train_loss_curv,
        "valid_loss_temp": valid_loss_temp, "valid_loss_curv": valid_loss_curv,
        "valid_f1": epoch_f1, "valid_accuracy": epoch_acc,
    })

epoch_history_df = pd.DataFrame(epoch_history)
epoch_history_df.to_csv(os.path.join(RESULT_DIR, "step4_sensor_calibration_history.csv"), index=False)

a_temp = a_temp_z / sigma_obs_temp
b_temp = b_temp_z - a_temp_z * mu_obs_temp / sigma_obs_temp
a_curv = a_curv_z / sigma_obs_curv
b_curv = b_curv_z - a_curv_z * mu_obs_curv / sigma_obs_curv

calib_summary = pd.DataFrame([
    {"sensor": "temperature", "a": a_temp, "b": b_temp},
    {"sensor": "curvature", "a": a_curv, "b": b_curv},
])
calib_summary.to_csv(os.path.join(RESULT_DIR, "step4_sensor_calibration.csv"), index=False)
print(f"[학습 완료] {N_EPOCHS} epoch 경사하강법 파인튜닝 -- "
      f"최종 calib_valid F1={epoch_history_df['valid_f1'].iloc[-1]:.3f}, "
      f"accuracy={epoch_history_df['valid_accuracy'].iloc[-1]:.3f}")
print(f"[보정] 온도센서: corrected = {a_temp:.3f} * obs + {b_temp:.3f}")
print(f"[보정] 굴곡센서: corrected = {a_curv:.3f} * obs + {b_curv:.3f}")
calib_summary

### B-7b — 센서 노이즈 크기에 따른 adaptive 이점 민감도 분석

F-3 3번 향후 개선 과제로 제안한 센서 노이즈 민감도 분석입니다. B-6은 온도·굴곡 센서 노이즈 표준편차를 각각 3.0°C, 1.5도로 고정했는데, 이 값이 커지면 adaptive의 이점(baseline 대비 F1 격차)이 줄어드는지 배율(0~5배)을 스윕하며 확인합니다.

**결과(SIMULATED).** Explode(굴곡) 과제의 F1 격차는 노이즈 배율 0배(무노이즈)에서 0.159였다가, 0.5배~2배 구간에서는 0.147 안팎으로 유지되고, 3배에서 0.136, 5배(온도 15°C·굴곡 7.5도)에서는 0.052까지 떨어집니다 — 노이즈가 커질수록 adaptive의 이점이 줄어든다는 가설이 뒷받침되었습니다. 다만 5배에서도 격차가 완전히 0이 되지는 않아, 이 정도로 큰 노이즈에서도 adaptive가 baseline보다 약간은 낫다는 점도 확인됩니다. Bad(열입력) 과제는 모든 노이즈 배율에서 격차가 정확히 0(둘 다 F1=1.0)이었는데, 이는 D-1에서 이미 확인한 것처럼 이번 TEST 표본 구성상 Bad 판정이 온도 보정 품질에 민감하지 않기 때문으로 해석됩니다.

In [ ]:
# ==========================================
# [B-7b, 신설·미실행] 센서 노이즈 크기 -> adaptive 이점(F1 격차) 민감도 스윕
# ==========================================
# 주의: SIMULATED 데이터 기반 분석이다. B-6의 노이즈 표준편차(온도 3.0°C, 굴곡 1.5도)를 배율로
# 바꿔가며 B-6(노이즈 주입)+B-7(경사하강 보정)+B-8(TEST 적용)을 재현한다. sim_temp_dev_C/
# sim_curvature_dev_deg(B-5, 노이즈와 무관한 SIMULATED 참값)는 그대로 두고 obs_* 노이즈만
# 매 스윕 지점마다 새로 뽑는다. baseline/true_*는 test_df 컬럼에 의존하지 않고 이 셀 안에서
# 직접 계산해, B-8 실행 여부와 무관하게 이 셀 하나만으로 동작하도록 했다.

BASE_TEMP_NOISE_STD = 3.0
BASE_CURV_NOISE_STD = 1.5
NOISE_SCALES_B7B = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]


def _sweep_one_noise_level(temp_noise_std, curv_noise_std, seed):
    local_rng = np.random.default_rng(seed)

    valid_local = valid_df.copy()
    test_local = test_df.copy()
    for df_part in (valid_local, test_local):
        n = len(df_part)
        df_part["obs_temp_dev_C"] = df_part["sim_temp_dev_C"] + local_rng.normal(0, temp_noise_std, size=n)
        df_part["obs_curvature_dev_deg"] = (
            df_part["sim_curvature_dev_deg"] + local_rng.normal(0, curv_noise_std, size=n))

    calib_train_local, calib_valid_local = train_test_split(
        valid_local, test_size=0.3, stratify=valid_local["category"], random_state=RANDOM_STATE)
    calib_train_local = calib_train_local.reset_index(drop=True)

    mu_t, sigma_t = standardize(calib_train_local["obs_temp_dev_C"].values)
    mu_c, sigma_c = standardize(calib_train_local["obs_curvature_dev_deg"].values)
    z_t = (calib_train_local["obs_temp_dev_C"].values - mu_t) / sigma_t
    z_c = (calib_train_local["obs_curvature_dev_deg"].values - mu_c) / sigma_c
    true_t = calib_train_local["sim_temp_dev_C"].values
    true_c = calib_train_local["sim_curvature_dev_deg"].values

    a_t_z, b_t_z = 0.0, float(true_t.mean())
    a_c_z, b_c_z = 0.0, float(true_c.mean())
    for epoch in range(1, N_EPOCHS + 1):
        lr = LR0 * (LR_DECAY ** (epoch - 1))
        resid_t = (a_t_z * z_t + b_t_z) - true_t
        a_t_z -= lr * 2.0 * np.mean(resid_t * z_t)
        b_t_z -= lr * 2.0 * np.mean(resid_t)
        resid_c = (a_c_z * z_c + b_c_z) - true_c
        a_c_z -= lr * 2.0 * np.mean(resid_c * z_c)
        b_c_z -= lr * 2.0 * np.mean(resid_c)

    a_t, b_t = a_t_z / sigma_t, b_t_z - a_t_z * mu_t / sigma_t
    a_c, b_c = a_c_z / sigma_c, b_c_z - a_c_z * mu_c / sigma_c

    corrected_temp_test = a_t * test_local["obs_temp_dev_C"] + b_t
    corrected_curv_test = a_c * test_local["obs_curvature_dev_deg"] + b_c
    Q_adapt_test = (
        (test_local["avg_current_A"] ** 2)
        * (1.0 + ALPHA_R_PER_C * corrected_temp_test)
        * test_local["weld_time_s"])
    tilt_adapt_test = test_local["angle_deg"] + corrected_curv_test

    adaptive_bad = (bad_rate_logistic_model(Q_adapt_test, Q_min_train, k_train) > 0.5).astype(int)
    adaptive_explode = (tilt_adapt_test > EXPLODE_TILT_THRESHOLD_DEG).astype(int)
    baseline_bad = (bad_rate_logistic_model(test_local["heat_input_proxy"], Q_min_train, k_train) > 0.5).astype(int)
    baseline_explode = (test_local["angle_deg"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)
    true_bad = (bad_rate_logistic_model(test_local["Q_effective_true"], Q_min_train, k_train) > 0.5).astype(int)
    true_explode = (test_local["effective_tilt_true_deg"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

    rows = []
    for task, y_true, y_adapt, y_base in [
        ("Bad(열입력)", true_bad, adaptive_bad, baseline_bad),
        ("Explode(굴곡)", true_explode, adaptive_explode, baseline_explode),
    ]:
        f1_adapt = f1_score(y_true, y_adapt, pos_label=1, zero_division=0)
        f1_base = f1_score(y_true, y_base, pos_label=1, zero_division=0)
        rows.append({
            "task": task, "temp_noise_std": temp_noise_std, "curv_noise_std": curv_noise_std,
            "adaptive_f1": f1_adapt, "baseline_f1": f1_base, "gap_f1": f1_adapt - f1_base,
        })
    return rows


sweep_rows_b7b = []
for i, scale in enumerate(NOISE_SCALES_B7B):
    rows = _sweep_one_noise_level(BASE_TEMP_NOISE_STD * scale, BASE_CURV_NOISE_STD * scale, seed=RANDOM_STATE + i)
    for r in rows:
        r["noise_scale"] = scale
    sweep_rows_b7b.extend(rows)

sweep_df_b7b = pd.DataFrame(sweep_rows_b7b)
sweep_df_b7b.to_csv(os.path.join(RESULT_DIR, "step4b_noise_sensitivity_sweep.csv"), index=False)

fig, ax = plt.subplots(figsize=(7, 5))
for task, sub in sweep_df_b7b.groupby("task"):
    ax.plot(sub["noise_scale"], sub["gap_f1"], marker="o", label=task)
ax.axhline(0, color="gray", ls=":")
ax.set_xlabel(f"노이즈 배율 (기준: 온도 {BASE_TEMP_NOISE_STD}°C, 굴곡 {BASE_CURV_NOISE_STD}도)")
ax.set_ylabel("F1 격차 (adaptive - baseline)")
ax.set_title("[신설·미실행, SIMULATED] 센서 노이즈 크기에 따른 adaptive 이점 민감도")
ax.legend()
ax.grid(True, ls=":", alpha=0.4)
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "noise_sensitivity_sweep.png"))
plt.show()
sweep_df_b7b

### B-8 — 예측기 3종 정의 및 TEST 적용 (참값 / baseline / adaptive)

- **참값(ground truth)**: 실제 관측된 결과가 아니라, 이런 편차가 실제로 있었다면 어떤 결과였을지에
  대한 물리모델 기반 가상 시나리오입니다.
- **baseline**: 편차를 전혀 모른 채 표준값(R=1, 편차 0)을 그대로 적용합니다.
- **adaptive**: 노이즈가 섞인 센서 관측값에 B-7의 보정을 적용해 판단합니다.

In [ ]:
# ==========================================
# [B-8] 예측기 3종(참값 / baseline / adaptive) 정의 및 TEST 적용
# ==========================================
test_df["true_bad_risk"] = (bad_rate_logistic_model(
    test_df["Q_effective_true"], Q_min_train, k_train) > 0.5).astype(int)
test_df["true_explode_risk"] = (test_df["effective_tilt_true_deg"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

# baseline -- 편차를 모르므로 원래의 heat_input_proxy(R=1 가정)와 angle_deg를 그대로 사용
test_df["baseline_bad_risk"] = (bad_rate_logistic_model(
    test_df["heat_input_proxy"], Q_min_train, k_train) > 0.5).astype(int)
test_df["baseline_explode_risk"] = (test_df["angle_deg"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

# adaptive -- VALID로 보정한 센서 관측값을 반영
test_df["corrected_temp_dev_C"] = a_temp * test_df["obs_temp_dev_C"] + b_temp
test_df["corrected_curvature_dev_deg"] = a_curv * test_df["obs_curvature_dev_deg"] + b_curv
test_df["R_eff_adapt"] = 1.0 + ALPHA_R_PER_C * test_df["corrected_temp_dev_C"]
test_df["Q_adapt"] = (test_df["avg_current_A"] ** 2) * test_df["R_eff_adapt"] * test_df["weld_time_s"]
test_df["tilt_adapt"] = test_df["angle_deg"] + test_df["corrected_curvature_dev_deg"]

test_df["adaptive_bad_risk"] = (bad_rate_logistic_model(
    test_df["Q_adapt"], Q_min_train, k_train) > 0.5).astype(int)
test_df["adaptive_explode_risk"] = (test_df["tilt_adapt"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

test_df.to_csv(os.path.join(RESULT_DIR, "step5_test_predictions.csv"), index=False)
print(f"[예측 완료] TEST {len(test_df)}건에 대해 참값/baseline/adaptive 3종 예측 생성")
test_df[["sample_id", "true_bad_risk", "baseline_bad_risk", "adaptive_bad_risk",
         "true_explode_risk", "baseline_explode_risk", "adaptive_explode_risk"]].head(10)


## C. 데이터셋 시각화

### C-1 — 분할 및 합성 변수 분포

7:2:1 분할 크기와, valid·test에 주입한 합성(SIMULATED) 편차의 분포를 확인합니다.

In [ ]:
# ==========================================
# [C-1] 분할 크기 및 합성 변수 분포 시각화
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=120)

axes[0].bar(split_summary["split"], split_summary["n"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axes[0].set_title("분할 크기 (7:2:1)")
axes[0].set_xlabel("split"); axes[0].set_ylabel("샘플 수")
for i, n in enumerate(split_summary["n"]):
    axes[0].text(i, n + 3, str(n), ha="center", fontweight="bold")

axes[1].hist(valid_df["sim_temp_dev_C"], bins=15, alpha=0.6, label="valid", color="#1f77b4")
axes[1].hist(test_df["sim_temp_dev_C"], bins=15, alpha=0.6, label="test", color="#d62728")
axes[1].set_title("합성(SIMULATED) 표면온도 편차 분포")
axes[1].set_xlabel("sim_temp_dev_C (°C)"); axes[1].set_ylabel("빈도")
axes[1].legend()

axes[2].hist(valid_df["sim_curvature_dev_deg"], bins=15, alpha=0.6, label="valid", color="#1f77b4")
axes[2].hist(test_df["sim_curvature_dev_deg"], bins=15, alpha=0.6, label="test", color="#d62728")
axes[2].set_title("합성(SIMULATED) 굴곡 편차 분포")
axes[2].set_xlabel("sim_curvature_dev_deg (도)"); axes[2].set_ylabel("빈도")
axes[2].set_ylim(axes[1].get_ylim())  # 2열(온도편차)과 세로축 범위를 통일해 빈도를 직접 비교 가능하게 함
axes[2].legend()

fig.suptitle("분할 크기 및 합성 편차 분포 (SIMULATED 변수)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "split_and_synthetic_distributions.png"))
plt.show()


**결과 해석.** `result_rsw_adaptive/step1_split_summary.csv`에 저장된 실제 분할 결과는
train 345건(Bad 15건, Explode 22건), valid 98건(Bad 4건, Explode 6건), test 50건(Bad 2건,
Explode 3건)입니다. 전체 493건 대비 약 70.0% : 19.9% : 10.1%로 목표한 7:2:1 비율과 거의
일치하며, 층화 분할 덕분에 표본이 가장 희소한 test에서도 Bad·Explode가 각각 최소 2건 이상
유지되어 이후 D-1의 precision·recall·F1 계산이 정의되지 않는 문제(분모가 0이 되는 경우)를
피할 수 있었습니다.

2열(표면온도 편차)과 3열(굴곡 편차) 히스토그램은 valid·test에 주입한 `sim_temp_dev_C`
(균등분포 [-15, 45]°C)와 `sim_curvature_dev_deg`(균등분포 [0, 20]°) 표본의 실제 실현값
분포를 보여줍니다. 두 변수 모두 정의 구간 전역에 걸쳐 대체로 고르게 분포하는지 확인하는 것이
목적이며, 특정 구간에 표본이 몰려 있다면 후속 회귀(B-7)의 보정 성능이 편향될 수 있습니다.
3열의 세로축 범위를 2열과 통일한 것은 두 히스토그램의 절대 빈도를 시각적으로 직접 비교할 수
있도록 하기 위함입니다.

## D. 데이터 시각화 및 해석

### D-1 — Baseline vs Adaptive 성능 지표 비교

TEST 세트에서 baseline과 adaptive의 accuracy·precision·recall·F1을 Bad(열입력)·Explode(굴곡)
과제별로 각각 비교합니다. 두 결함은 서로 다른 메커니즘이므로 하나로 합치지 않습니다.

In [ ]:
# ==========================================
# [D-1] Baseline vs Adaptive 성능 지표 비교 (TEST, Bad/Explode 각각)
# ==========================================
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

metric_rows = []
for task, true_col, pred_prefix in [("Bad(열입력)", "true_bad_risk", "bad_risk"),
                                      ("Explode(굴곡)", "true_explode_risk", "explode_risk")]:
    for method in ("baseline", "adaptive"):
        m = compute_metrics(test_df[true_col], test_df[f"{method}_{pred_prefix}"])
        metric_rows.append({"task": task, "method": method, **m})

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(RESULT_DIR, "step6_baseline_vs_adaptive_metrics.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=120)
metric_names = ["accuracy", "precision", "recall", "f1"]
for ax, task in zip(axes, metrics_df["task"].unique()):
    sub = metrics_df[metrics_df["task"] == task]
    x = np.arange(len(metric_names))
    width = 0.35
    baseline_vals = sub[sub["method"] == "baseline"][metric_names].values.flatten()
    adaptive_vals = sub[sub["method"] == "adaptive"][metric_names].values.flatten()
    ax.bar(x - width / 2, baseline_vals, width, label="baseline(표준값 고정)", color="#7f7f7f")
    ax.bar(x + width / 2, adaptive_vals, width, label="adaptive(감지+보정)", color="#2ca02c")
    ax.set_xticks(x); ax.set_xticklabels(metric_names)
    ax.set_ylim(0, 1.05)
    ax.set_title(task)
    ax.legend(loc="lower right")
    ax.grid(True, axis="y", ls=":", alpha=0.4)

fig.suptitle("Baseline vs Adaptive 성능 비교 (TEST, 시뮬레이션 기반 가상 시나리오)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "baseline_vs_adaptive_metrics.png"))
plt.show()
metrics_df


**[재실행 확인 완료]** B-7을 미니배치 SGD로 바꾸고 재실행한 결과, 보정계수는 `a_temp/b_temp≈0.967/0.156`, `a_curv/b_curv≈0.925/0.749`로 이전(전체배치)과 소폭 달라졌지만, 아래 표의 TEST 지표는 **수치까지 완전히 동일**했습니다. 즉 이번 TEST 50건에서는 이 정도의 계수 변화로 판정 경계를 넘는 표본이 하나도 없었다는 뜻이며, D-1b의 부트스트랩 신뢰구간이 이 결과의 통계적 안정성을 뒷받침합니다(아래 참고).

**결과 해석.** `result_rsw_adaptive/step6_baseline_vs_adaptive_metrics.csv`의 실제 수치는
다음과 같습니다.

| 과제 | 방식 | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|---|
| Bad(열입력) | baseline | 1.000 | 1.000 | 1.000 | 1.000 |
| Bad(열입력) | adaptive | 1.000 | 1.000 | 1.000 | 1.000 |
| Explode(굴곡) | baseline | 0.760 | 1.000 | 0.707 | 0.829 |
| Explode(굴곡) | adaptive | 0.960 | 0.976 | 0.976 | 0.976 |

**Explode(굴곡) 과제에서는 adaptive의 개선이 뚜렷합니다.** baseline은 recall이 0.707에 그쳐
굴곡 위험이 실제로 존재하는 표본 중 약 29%를 놓쳤지만, adaptive는 recall을 0.976까지 끌어올려
정확도·F1 모두 baseline을 크게 앞섰습니다(구체적인 오탐·누락 건수는 D-2 혼동행렬 참고).

**Bad(열입력) 과제에서는 baseline과 adaptive의 차이가 전혀 없습니다(둘 다 완벽하게 일치).**
이는 오류가 아니라, 이번 TEST 표본(50건)에서 온도 편차가 만든 열입력 $Q$의 변동폭이 Bad
판정 경계($Q_{min}=1.209\times10^{6}\pm6.33\times10^{4}$, 오차비율 5.2%)를 넘어설 만큼
크지 않았기 때문으로 해석됩니다. 저항 온도계수 $\alpha=0.004$/°C와 편차 범위(-15~45°C)를
적용하면 $R_{eff}$는 약 0.94~1.18배 범위에서 변하는데, 이 TEST 표본들의 원래 $Q$ 값이 대체로
$Q_{min}$ 경계로부터 충분히 떨어져 있어 이 정도 변동으로는 판정이 뒤집히지 않은 것으로
보입니다. 즉 이번 결과는 "온도 편차는 항상 무해하다"는 뜻이 아니라, 이 특정 표본 구성에서
우연히 경계를 넘는 사례가 없었다는 뜻이며, F-3의 민감도 분석 제안과도 연결됩니다.

### D-1b — Bad·Explode 지표의 부트스트랩 신뢰구간

F-3 6번 향후 개선 과제입니다. TEST는 50건뿐이고 그중 Bad 양성은 2건뿐이라, D-1의 점추정값 하나만으로는 baseline과 adaptive의 차이가 통계적으로 의미 있는지 알기 어렵습니다. TEST를 복원추출(bootstrap, 2000회)로 반복 재구성해 accuracy·precision·recall·F1의 95% 신뢰구간을 계산합니다.

**결과(SIMULATED).** Bad(열입력) 과제는 accuracy는 [1.00, 1.00]으로 안정적이지만, precision·recall·F1의 신뢰구간은 [0.00, 1.00]으로 사실상 전체 범위를 덮습니다 — 양성이 2건뿐이라 복원추출 시 양성이 하나도 안 뽑히는 경우가 흔해서입니다. 즉 D-1의 "Bad는 baseline·adaptive가 완전히 동일"이라는 결과는 **점추정으로는 맞지만, 이 지표만으로 두 방식의 우열을 통계적으로 단정할 수 없다**는 뜻입니다.

**Explode(굴곡) 과제는 반대로 신뢰구간이 서로 겹치지 않습니다.** baseline accuracy [0.64, 0.88] vs adaptive accuracy [0.90, 1.00] — 표본 재구성에 따른 불확실성을 감안해도 adaptive가 더 우수하다는 결론이 안정적으로 유지됩니다. F1도 baseline [0.72, 0.92] vs adaptive [0.94, 1.00]로 마찬가지입니다. 즉 D-1의 Explode 개선은 이번 TEST 표본 하나의 우연이 아니라 통계적으로도 근거 있는 결과입니다.

In [ ]:
# ==========================================
# [D-1b, 신설·미실행] Bad·Explode 지표 부트스트랩 95% 신뢰구간
# ==========================================
# 주의: SIMULATED 참값(true_bad_risk/true_explode_risk) 기준 지표에 대한 신뢰구간이다.
# D-1의 compute_metrics()를 그대로 재사용한다.
N_BOOTSTRAP_D1B = 2000
rng_d1b = np.random.default_rng(RANDOM_STATE)
n_test_d1b = len(test_df)

boot_rows_d1b = []
for task, true_col, pred_prefix in [("Bad(열입력)", "true_bad_risk", "bad_risk"),
                                      ("Explode(굴곡)", "true_explode_risk", "explode_risk")]:
    for method in ("baseline", "adaptive"):
        y_true_full = test_df[true_col].values
        y_pred_full = test_df[f"{method}_{pred_prefix}"].values
        boot_metrics = {"accuracy": [], "precision": [], "recall": [], "f1": []}
        for _ in range(N_BOOTSTRAP_D1B):
            idx = rng_d1b.integers(0, n_test_d1b, size=n_test_d1b)
            m = compute_metrics(y_true_full[idx], y_pred_full[idx])
            for k, v in m.items():
                boot_metrics[k].append(v)
        row = {"task": task, "method": method}
        for k, vals in boot_metrics.items():
            arr = np.array(vals)
            row[f"{k}_median"] = np.median(arr)
            row[f"{k}_ci_lo"] = np.percentile(arr, 2.5)
            row[f"{k}_ci_hi"] = np.percentile(arr, 97.5)
        boot_rows_d1b.append(row)

metrics_ci_df = pd.DataFrame(boot_rows_d1b)
metrics_ci_df.to_csv(os.path.join(RESULT_DIR, "step6b_metrics_bootstrap_ci.csv"), index=False)
print("[D-1b, 신설·미실행] 부트스트랩 신뢰구간 계산 코드 -- 로컬에서 실행해 결과를 확인하십시오.")
metrics_ci_df

### D-2 — 혼동행렬 비교

D-1에서 요약된 accuracy·precision·recall·F1이 실제로 어떤 예측/오답 조합에서 나온 것인지 TP·FP·FN·TN 단위로 분해해 확인합니다. Bad·Explode 두 과제, baseline·adaptive 두 방식을 조합한 4개 혼동행렬을 나란히 제시합니다.

In [ ]:
# ==========================================
# [D-2] 혼동행렬 비교 (Bad/Explode x baseline/adaptive)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(10, 9), dpi=120)
cm_specs = [
    ("Bad(열입력) x baseline", "true_bad_risk", "baseline_bad_risk", axes[0, 0]),
    ("Bad(열입력) x adaptive", "true_bad_risk", "adaptive_bad_risk", axes[0, 1]),
    ("Explode(굴곡) x baseline", "true_explode_risk", "baseline_explode_risk", axes[1, 0]),
    ("Explode(굴곡) x adaptive", "true_explode_risk", "adaptive_explode_risk", axes[1, 1]),
]
for title, true_col, pred_col, ax in cm_specs:
    cm = confusion_matrix(test_df[true_col], test_df[pred_col], labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["안전(0)", "위험(1)"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["안전(0)", "위험(1)"])
    ax.set_xlabel("예측"); ax.set_ylabel("참값(시뮬레이션)")
    ax.set_title(title)

    # 칸 사이 경계선(회색)
    ax.set_xticks(np.arange(-0.5, 2, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 2, 1), minor=True)
    ax.grid(which="minor", color="gray", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)

    cm_total = cm.sum()
    for i in range(2):
        for j in range(2):
            pct = cm[i, j] / cm_total * 100 if cm_total > 0 else 0.0
            ax.text(j, i, f"{cm[i, j]}\n({pct:.1f}%)", ha="center", va="center", fontweight="bold",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")

fig.suptitle("혼동행렬: baseline vs adaptive (TEST)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "confusion_matrices.png"))
plt.show()


**결과 해석.** 4개 혼동행렬은 D-1 지표의 산출 근거를 그대로 보여줍니다(TEST 50건 기준,
TP/FP/FN/TN).

| 과제 × 방식 | TP | FP | FN | TN |
|---|---|---|---|---|
| Bad × baseline | 2 | 0 | 0 | 48 |
| Bad × adaptive | 2 | 0 | 0 | 48 |
| Explode × baseline | 29 | 0 | 12 | 9 |
| Explode × adaptive | 40 | 1 | 1 | 8 |

Bad 과제는 baseline·adaptive 모두 완전히 동일한 혼동행렬을 보여 D-1에서 확인한 지표 동일성과
정확히 일치합니다. Explode 과제는 `true_explode_risk=1`(위험)인 표본이 TEST 50건 중 41건으로
대다수를 차지하는데, 이는 굴곡 편차가 균등분포 [0°, 20°]의 항상 양수인 값이라 `angle_deg`가
0°인 표본조차 상당수가 임계값(7.5°)을 넘기기 때문입니다. baseline은 이 41건 중 12건을
놓쳤지만(FN=12), adaptive는 1건만 놓쳐(FN=1) 굴곡 위험 탐지 성능이 크게 개선되었습니다. 다만
FP도 0에서 1로 소폭 늘었는데, 이는 센서 노이즈로 인해 실제로는 안전한 표본을 위험으로
오판정한 경우이며, "센서를 신뢰하는 대가로 완벽한 정밀도를 100% 보장하지는 못한다"는
adaptive 방식의 현실적인 특성을 보여줍니다.

### D-3 — 이미지 보유 99건 확장 시연 데이터 구성

D-1/D-2의 baseline vs adaptive 성능 비교는 통계적 타당성을 위해 반드시 공식 TEST 분할(약 50건)만
사용했지만, 그중 실제 IR 이미지가 있는 표본은 일부뿐이라 대시보드 애니메이션에서 대부분 이미지가
비어 있는 문제가 있었습니다. 이 절에서는 이미 학습을 마친 표준값(`Q_min_train`, `k_train`,
`EXPLODE_TILT_THRESHOLD_DEG`)과 VALID로 보정한 센서 계수(`a_temp`, `b_temp`, `a_curv`, `b_curv`)를
"학습이 끝난 고정 모델"로 간주하고, IR 이미지가 있는 493건 중 99건 전체(원래 train/valid/test
분할과 무관하게)에 동일하게 적용해보는 시연용 확장 데이터셋을 구성합니다. 여기서 "IR 이미지"는
적외선 열화상 카메라로 찍은 뒤 이미 Jet 색상 팔레트로 렌더링된 사진입니다(절대 온도 계측값이
아니라 상대적 열 분포를 보여주는 시각 자료이며, 원본 데이터셋 493건 중 99건에만 제공됩니다).

**이 데이터셋은 D-1/D-2가 보고하는 공식 TEST 지표와는 표본 구성이 다른, 대시보드 시연 전용
데이터입니다.** 물리모델과 보정계수는 이미 확정된 값을 재사용할 뿐 이 단계에서 추가로 학습하지
않으므로, "이미 배포된 시스템을 새 입력에 적용해본다"는 것과 동일한 성격입니다.

In [ ]:
# ==========================================
# [D-3] 이미지 보유 99건 확장 시연 데이터 구성
# ==========================================
IMG_DIR_D3 = os.path.join("Data", "Resistance Spot Welding Insights", "ir_images")
ir_ids_d3 = set()
if os.path.isdir(IMG_DIR_D3):
    for fname in os.listdir(IMG_DIR_D3):
        if fname.lower().endswith(".jpg") and fname.startswith("IR_"):
            ir_ids_d3.add(int(fname.split("_")[1].split(".")[0]))

demo99_df = agg[agg["sample_id"].isin(ir_ids_d3)].sort_values("heat_input_proxy").reset_index(drop=True)
print(f"[안내] IR 이미지 보유 표본 {len(demo99_df)}건 전체를 시연용으로 사용합니다 "
      f"(공식 TEST 분할과 별개 -- D-1/D-2 지표는 그대로 TEST 분할 기준입니다).")

rng_demo = np.random.default_rng(43)  # 공식 실험(RANDOM_STATE=42)과 구분되는 시연 전용 시드
n_demo = len(demo99_df)

demo99_df["sim_temp_dev_C"] = rng_demo.uniform(-15, 45, size=n_demo)
demo99_df["sim_curvature_dev_deg"] = rng_demo.uniform(0, 20, size=n_demo)
demo99_df["obs_temp_dev_C"] = demo99_df["sim_temp_dev_C"] + rng_demo.normal(0, 3.0, size=n_demo)
demo99_df["obs_curvature_dev_deg"] = demo99_df["sim_curvature_dev_deg"] + rng_demo.normal(0, 1.5, size=n_demo)

demo99_df["R_eff_true"] = 1.0 + ALPHA_R_PER_C * demo99_df["sim_temp_dev_C"]
demo99_df["Q_effective_true"] = (demo99_df["avg_current_A"] ** 2) * demo99_df["R_eff_true"] * demo99_df["weld_time_s"]
demo99_df["effective_tilt_true_deg"] = demo99_df["angle_deg"] + demo99_df["sim_curvature_dev_deg"]

demo99_df["true_bad_risk"] = (bad_rate_logistic_model(
    demo99_df["Q_effective_true"], Q_min_train, k_train) > 0.5).astype(int)
demo99_df["true_explode_risk"] = (demo99_df["effective_tilt_true_deg"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

demo99_df["baseline_bad_risk"] = (bad_rate_logistic_model(
    demo99_df["heat_input_proxy"], Q_min_train, k_train) > 0.5).astype(int)
demo99_df["baseline_explode_risk"] = (demo99_df["angle_deg"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

demo99_df["corrected_temp_dev_C"] = a_temp * demo99_df["obs_temp_dev_C"] + b_temp
demo99_df["corrected_curvature_dev_deg"] = a_curv * demo99_df["obs_curvature_dev_deg"] + b_curv
demo99_df["R_eff_adapt"] = 1.0 + ALPHA_R_PER_C * demo99_df["corrected_temp_dev_C"]
demo99_df["Q_adapt"] = (demo99_df["avg_current_A"] ** 2) * demo99_df["R_eff_adapt"] * demo99_df["weld_time_s"]
demo99_df["tilt_adapt"] = demo99_df["angle_deg"] + demo99_df["corrected_curvature_dev_deg"]

demo99_df["adaptive_bad_risk"] = (bad_rate_logistic_model(
    demo99_df["Q_adapt"], Q_min_train, k_train) > 0.5).astype(int)
demo99_df["adaptive_explode_risk"] = (demo99_df["tilt_adapt"] > EXPLODE_TILT_THRESHOLD_DEG).astype(int)

demo99_df.to_csv(os.path.join(RESULT_DIR, "step6_demo99_predictions.csv"), index=False)
print(f"[완료] 시연용 {n_demo}건에 대해 참값/baseline/adaptive 예측 생성 완료")
demo99_df[["sample_id", "sim_temp_dev_C", "sim_curvature_dev_deg",
           "true_bad_risk", "baseline_bad_risk", "adaptive_bad_risk"]].head(5)

### D-4 — 실시간 종합 모니터링 대시보드 (3×3 애니메이션)

`1_rsw_optimization.ipynb`의 D-10 대시보드(축척바 IR 이미지, 컬러바, 피팅 곡선 위 실시간 포인트, 너겟·인장강도 예측)를 이 노트북의 baseline vs adaptive 비교 구조에 맞춰 3×3으로 확장했습니다. D-3에서 구성한 IR 이미지 보유 99건을 순서대로 재생합니다.

**핵심 용어**

| 용어 | 정의 |
|---|---|
| 열입력 $Q$ | 저항용접의 발열량. Joule 가열 법칙 $Q = I^2 R_{eff} t$ (전류 $I$, 유효 접촉저항 $R_{eff}$, 통전시간 $t$) |
| $R_{eff}$ | 온도에 따라 변하는 유효 접촉저항. $R_{eff} = 1 + \alpha \Delta T$ (철강계 저항온도계수 근사값 $\alpha=0.004/°C$, 이 데이터셋에서 직접 측정된 값은 아님) |
| baseline / adaptive | baseline은 $R_{eff}=1$로 가정한(보정 없는) 예측, adaptive는 B-7에서 학습한 보정계수로 온도·굴곡 편차를 반영한 예측 |
| Bad(미용착) | 열입력 부족으로 너겟이 충분히 자라지 못한 불량 |
| Explode(팽출) | 전극각도가 틀어져 과도한 압력이 걸리며 스패터가 튀는 불량 |
| 너겟 지름 | 용접부에서 실제로 녹았다 굳은 금속 버튼의 지름. 접합강도를 좌우하는 핵심 품질지표 |
| SIMULATED | 실측이 아니라 물리적으로 동기부여된 가정으로 합성 주입한 값(B-5~B-8) |

**패널 구성**

| 위치 | 내용 |
|---|---|
| 1행1열 | 실측 IR(적외선 열화상) 이미지 + 축척바(10mm) + 컬러바 + 좌상단 샘플번호·우상단 전극각도 |
| 1행2열 | baseline vs adaptive 누적(rolling) 정확도 |
| 1행3열 | 실측 공정 위상도(전류×통전시간), 전극각도별 안전/위험 배경 |
| 2행1열 | 표면온도 편차 -- 실제(SIMULATED) vs 센서 감지값, 샘플별 위험/경계/안전 배경(세로축 -25~65℃, y=0 기준선) |
| 2행2열 | 굴곡 편차 -- 실제(SIMULATED) vs 센서 감지값, 샘플별 위험/경계/안전 배경(세로축 2행3열과 통일, y=0 기준선) |
| 2행3열 | 공정 위상도($Q$ × 유효 접촉각) -- 참값(SIMULATED)·baseline·adaptive 3점, 99건 배경 산점, y=0 기준선 |
| 3행1열 | 인장강도 예측(너겟지름 → 인장강도 선형회귀) -- 3점 |
| 3행2열 | 너겟 지름 성장 곡선($Q$ → 너겟지름) -- 3점, 가로축 0~2.0e7 |
| 3행3열 | Bad 확률 피팅 곡선($Q$ → Bad 확률) -- 3점, 가로축 0~2.0e7 |

**피팅 곡선의 수식과 근거**

1. **Bad 확률(B-3)**: $p_{bad}(Q) = \dfrac{1}{1 + e^{k(Q - Q_{min})}}$ — TRAIN 표본에 `scipy.curve_fit`으로 피팅한 감소형 로지스틱(시그모이드). $Q < Q_{min}$(열입력 부족)일수록 미용착 확률이 1에 가까워지고, $Q > Q_{min}$이면 0에 가까워지며, $Q=Q_{min}$에서 정확히 0.5. 이 $Q_{min}$과 표준오차(±)가 1행3열·2행3열 배경의 위험/경계/안전 구간 판정에도 그대로 쓰입니다.
2. **너겟 지름 성장(D-2 재사용)**: $D(Q) = D_0 + (D_{max}-D_0)(1-e^{-Q/\tau})$ — 열입력이 늘수록 녹는 금속량도 늘지만, 판재 두께와 전극 냉각으로 결국 최대 지름 $D_{max}$에 가까워지며 성장이 둔화되는 포화형(saturating) 곡선. 저항용접 너겟 성장의 전형적 형태를 반영한 가정입니다.
3. **인장강도 예측**: $N = a \cdot D + b$ (너겟지름-인장강도 선형회귀, `scipy.linregress`로 493건에서 직접 회귀). 너겟 단면적(≈접합 면적)이 클수록 파단에 필요한 힘이 커진다는 관계를 실측값으로 근사한 것입니다.
4. **Explode 임계값(B-4)**: 접촉각 두 실측 수준(0°/15°)의 단순 중간값인 7.5도를 고정 규칙으로 사용(데이터로 피팅한 값이 아님). 접촉각이 이 값을 넘으면 팽출 위험 구간으로 표시됩니다.

3행2열(너겟)은 `result_rsw/step4_nugget_fit_params.csv`, 3행3열(Bad 확률)은 B-3에서 이미 확정된 피팅 결과를 그대로 재사용합니다. 3행1열(인장강도)만 이 노트북에서 `agg` 실측값에 즉석으로 선형회귀를 계산하며, 셋 다 반복적인 모델 학습을 새로 수행하지는 않습니다. 3행2열·3행3열의 에러바는 실제 피팅에 쓰인 구간별 평균±SEM(`step2_binned_stats.csv`)입니다. D-1/D-2의 공식 TEST 지표(약 50건)와는 표본 구성이 다르다는 점(99건)은 D-3와 동일합니다.

**2행1열·2행2열 배경(신설)**: 이 두 패널의 위험/경계/안전 배경은 새로 만든 임계값이 아니라, 위 Bad 확률·Explode 임계값을 각 샘플 조건에 대해 역산한 것입니다. 2행1열은 그 샘플의 전류·통전시간을 고정한 채 $Q=I^2(1+\alpha\Delta T)t=Q_{min}\pm$오차를 $\Delta T$에 대해 풀고, 2행2열은 그 샘플의 전극각도를 고정한 채 유효 접촉각이 `EXPLODE_TILT_THRESHOLD_DEG±TILT_BAND_DEG`를 넘는 $\Delta tilt$ 경계를 구합니다. 그 결과 "실제(SIMULATED)" 선이 y=0(편차 없음)에 있을 때의 색은 1행3열의 배경색과, 실제 값 위치의 색은 2행3열 배경점의 색과 정확히 일치하도록 설계했습니다.

In [ ]:
# ==========================================
# [D-4] 실시간 종합 모니터링 대시보드 (3x3 애니메이션) -- 이미지 보유 99건 전체
# ==========================================
import cv2
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from scipy.stats import linregress

n_frames_d4 = len(demo99_df)

# 주의: true_bad_risk·true_explode_risk는 실측이 아니라 SIMULATED 참값(B-8)이며, 아래
# baseline/adaptive 누적 정확도는 이 SIMULATED 참값을 정답으로 삼아 계산한 것이다(A-3).
baseline_correct_d4 = ((demo99_df["baseline_bad_risk"] == demo99_df["true_bad_risk"]) &
                        (demo99_df["baseline_explode_risk"] == demo99_df["true_explode_risk"])).astype(int).values
adaptive_correct_d4 = ((demo99_df["adaptive_bad_risk"] == demo99_df["true_bad_risk"]) &
                        (demo99_df["adaptive_explode_risk"] == demo99_df["true_explode_risk"])).astype(int).values
baseline_rolling_acc_d4 = np.cumsum(baseline_correct_d4) / (np.arange(n_frames_d4) + 1)
adaptive_rolling_acc_d4 = np.cumsum(adaptive_correct_d4) / (np.arange(n_frames_d4) + 1)

temp_true_arr = demo99_df["sim_temp_dev_C"].values
temp_obs_arr = demo99_df["obs_temp_dev_C"].values
curv_true_arr = demo99_df["sim_curvature_dev_deg"].values
curv_obs_arr = demo99_df["obs_curvature_dev_deg"].values

# 주의: Q_true_arr·tilt_true_arr(Q_effective_true, effective_tilt_true_deg)와 Q_adapt_arr·
# tilt_adapt_arr는 전부 SIMULATED 값이다. 실측 원값은 Q_base_arr(heat_input_proxy)뿐이다(A-3).
Q_true_arr = demo99_df["Q_effective_true"].values
Q_base_arr = demo99_df["heat_input_proxy"].values
Q_adapt_arr = demo99_df["Q_adapt"].values
tilt_true_arr = demo99_df["effective_tilt_true_deg"].values
tilt_base_arr = demo99_df["angle_deg"].values
tilt_adapt_arr = demo99_df["tilt_adapt"].values

# ---------- 1행1열 배경: 축척바 측정 (1_rsw_optimization.ipynb의 D-10과 동일한 방식) ----------
SAT_THRESHOLD_D4 = 0.15
VAL_THRESHOLD_D4 = 0.8
MIN_LINE_WIDTH_FRAC_D4 = 0.3


def to_hsv_d4(img_rgb: np.ndarray):
    hsv_u8 = cv2.cvtColor(img_rgb[..., :3].astype(np.uint8), cv2.COLOR_RGB2HSV)
    hue_deg = hsv_u8[..., 0].astype(np.float64) * (360.0 / 179.0)
    sat = hsv_u8[..., 1].astype(np.float64) / 255.0
    val = hsv_u8[..., 2].astype(np.float64) / 255.0
    return hue_deg, sat, val


def measure_scalebar_length_px_d4(img_rgb: np.ndarray):
    """1_rsw_optimization.ipynb D-10의 measure_scalebar_length_px와 동일한 로직. 원본(크롭 전)
    이미지에서 측정해야 축척바 길이를 정확히 잴 수 있다."""
    _, sat, val = to_hsv_d4(img_rgb)
    white_mask = (sat < SAT_THRESHOLD_D4) & (val > VAL_THRESHOLD_D4)
    n_labels, labeled = cv2.connectedComponents(white_mask.astype(np.uint8), connectivity=8)
    best_w = None
    for label_id in range(1, n_labels):
        ys, xs = np.nonzero(labeled == label_id)
        bbox_w = xs.max() - xs.min() + 1
        if bbox_w < MIN_LINE_WIDTH_FRAC_D4 * img_rgb.shape[1]:
            continue
        if best_w is None or bbox_w > best_w:
            best_w = bbox_w
    return best_w


MAX_LINE_THICKNESS_FRAC_D4 = 0.15


def find_scalebar_crop_row_d4(img_rgb: np.ndarray):
    """1_rsw_optimization.ipynb B-4의 find_scalebar_crop_row와 동일한 로직 -- 축척바 눈금선의
    (최상단 행, 두께)를 찾는다."""
    h, w = img_rgb.shape[:2]
    _, sat, val = to_hsv_d4(img_rgb)
    white_mask = (sat < SAT_THRESHOLD_D4) & (val > VAL_THRESHOLD_D4)
    n_labels, labeled = cv2.connectedComponents(white_mask.astype(np.uint8), connectivity=8)
    best = None
    for label_id in range(1, n_labels):
        ys, xs = np.nonzero(labeled == label_id)
        bbox_w = xs.max() - xs.min() + 1
        if bbox_w < MIN_LINE_WIDTH_FRAC_D4 * w:
            continue
        top_per_col, thickness_per_col = [], []
        for x in range(xs.min(), xs.max() + 1):
            col_ys = ys[xs == x]
            if col_ys.size == 0:
                continue
            y0 = int(col_ys.min())
            top_per_col.append(y0)
            run = 1
            while y0 + run < h and white_mask[y0 + run, x]:
                run += 1
            thickness_per_col.append(run)
        if not top_per_col:
            continue
        thickness = int(np.median(thickness_per_col))
        if thickness > MAX_LINE_THICKNESS_FRAC_D4 * bbox_w:
            continue
        top_row = int(np.min(top_per_col))
        if best is None or bbox_w > best[0]:
            best = (bbox_w, top_row, thickness)
    return (best[1], best[2]) if best else None


def crop_above_scalebar_d4(img_rgb: np.ndarray):
    """원본 사진에 이미 찍혀 있는 축척바를 잘라내고, 그 위쪽만 남긴다 -- 이렇게 해야 우리가 직접
    그리는 10mm 축척바가 사진에 원래 있던 것과 중복되지 않는다(1_rsw_optimization.ipynb와 동일 방식)."""
    found = find_scalebar_crop_row_d4(img_rgb)
    if found is None:
        return img_rgb, None
    top_row, thickness = found
    cutoff = max(top_row - thickness, 1)
    return img_rgb[:cutoff, :, :], cutoff


sample_orig_img_d4 = plt.imread(os.path.join(IMG_DIR_D3, f"IR_{int(demo99_df.iloc[0]['sample_id'])}.jpg"))
scalebar_px_10mm_d4 = measure_scalebar_length_px_d4(sample_orig_img_d4)
print(f"[스케일 보정] 축척바 길이 {scalebar_px_10mm_d4}px = 10mm")

frames_d4 = []
for sid in demo99_df["sample_id"]:
    raw_img_d4 = plt.imread(os.path.join(IMG_DIR_D3, f"IR_{int(sid)}.jpg"))
    cropped_img_d4, _ = crop_above_scalebar_d4(raw_img_d4)
    frames_d4.append(cropped_img_d4)

# ---------- 1행3열/2행3열 배경: 공정 위상도 + Bad 확률 피팅 곡선 (B-3/B-4 표준값 모델) ----------
Q_all_d4 = np.concatenate([Q_true_arr, Q_base_arr, Q_adapt_arr, agg["heat_input_proxy"].values])  # 근거 에러바(전체 493건 기준)까지 곡선이 덮도록 범위를 넓힌다
Q_lo_d4, Q_hi_d4 = Q_all_d4.min() * 0.9, Q_all_d4.max() * 1.1

# 3행2열(너겟)·3행3열(Bad 확률) 표시 범위: 실측 493건 전체 상한(Q_hi_d4)을 보기 좋게 반올림한
# 고정값. step1_aggregated_samples.csv가 바뀌어 Q_hi_d4가 이 값을 넘으면 아래에서 경고한다.
Q_XLIM_D4 = (0.0, 2.0e7)
if Q_hi_d4 > Q_XLIM_D4[1] * 1.05:
    print(f"[경고] 실측 데이터 상한({Q_hi_d4:.2e})이 3행2열/3행3열 고정 표시범위"
          f"({Q_XLIM_D4[1]:.2e})를 초과합니다 -- Q_XLIM_D4 재조정 필요")
tilt_lo_d4, tilt_hi_d4 = -2.0, 32.0  # 세로축(유효 접촉각) 표시 범위 고정

QQ_d4, TT_d4 = np.meshgrid(np.linspace(Q_lo_d4, Q_hi_d4, 220), np.linspace(tilt_lo_d4, tilt_hi_d4, 220))
bad_zone_d4 = np.where(QQ_d4 < Q_min_train - Q_min_err_train, 0,
                        np.where(QQ_d4 < Q_min_train + Q_min_err_train, 1, 2))
TILT_BAND_DEG = 2.0  # EXPLODE_TILT_THRESHOLD_DEG 자체가 피팅값이 아닌 고정 규칙이므로,
                      # 경계 표시용으로 임의 부여한 시각화 폭이며 통계적 오차가 아니다
explode_zone_d4 = np.where(TT_d4 > EXPLODE_TILT_THRESHOLD_DEG + TILT_BAND_DEG, 0,
                            np.where(TT_d4 > EXPLODE_TILT_THRESHOLD_DEG - TILT_BAND_DEG, 1, 2))
combo_zone_d4 = np.minimum(bad_zone_d4, explode_zone_d4)  # 둘 중 더 위험한 쪽이 전체 구역을 결정
zone_cmap_d4 = ListedColormap(['#f8d7d7', '#ffe9a8', '#d5f2d5'])  # 위험/경계/안전 (D-9와 동일 팔레트)

Q_curve_d4 = np.linspace(Q_lo_d4, Q_hi_d4, 300)
bad_curve_d4 = bad_rate_logistic_model(Q_curve_d4, Q_min_train, k_train)

# ---------- 3행1열/3행2열 배경: 너겟 성장 곡선 + 인장강도 회귀 (1_rsw_optimization.ipynb 재사용) ----------
nugget_fit_pd = pd.read_csv(os.path.join("result_rsw", "step4_nugget_fit_params.csv")).iloc[0]

# 근거 데이터: 1_rsw_optimization.ipynb가 실제로 피팅에 사용한 구간별 평균±SEM(step2_binned_stats.csv)을
# 그대로 재사용한다 -- Bad 확률 곡선과 너겟 성장 곡선 모두 이 통계량에 피팅된 것이다.
bad_binned_d4 = pd.read_csv(os.path.join("result_rsw", "step2_binned_stats.csv"))

# 인장강도 회귀는 원 노트북에 구간화된 CSV가 없으므로, 동일한 관례(qcut 6구간, 평균±SEM)로
# agg의 실측 너겟지름-인장강도 관계를 직접 구간화한다.
pull_bins_d4 = pd.qcut(agg["nugget_diameter_mm"], q=6, duplicates="drop")
pull_binned_d4 = agg.groupby(pull_bins_d4, observed=True).agg(
    nugget_mean=("nugget_diameter_mm", "mean"),
    nugget_sem=("nugget_diameter_mm", "sem"),
    pull_mean=("pull_test_N", "mean"),
    pull_sem=("pull_test_N", "sem"),
).reset_index(drop=True)


def nugget_growth_model(Q, D0, Dmax, tau):
    return D0 + (Dmax - D0) * (1.0 - np.exp(-Q / tau))


def predict_nugget_d4(Q):
    return nugget_growth_model(Q, nugget_fit_pd["D0_opt"], nugget_fit_pd["Dmax_opt"], nugget_fit_pd["tau_opt"])


pull_lr_d4 = linregress(agg["nugget_diameter_mm"], agg["pull_test_N"])


def predict_pull_d4(nugget_d):
    return pull_lr_d4.slope * nugget_d + pull_lr_d4.intercept


nugget_curve_d4 = predict_nugget_d4(Q_curve_d4)
nugget_range_d4 = np.linspace(agg["nugget_diameter_mm"].min(), agg["nugget_diameter_mm"].max(), 200)
pull_curve_d4 = predict_pull_d4(nugget_range_d4)

nugget_true_arr = predict_nugget_d4(Q_true_arr)
nugget_base_arr = predict_nugget_d4(Q_base_arr)
nugget_adapt_arr = predict_nugget_d4(Q_adapt_arr)
pull_true_arr = predict_pull_d4(nugget_true_arr)
pull_base_arr = predict_pull_d4(nugget_base_arr)
pull_adapt_arr = predict_pull_d4(nugget_adapt_arr)


*(위 셀에서 배경 계산(1행1열 축척바, 각 패널의 곡선·zone 데이터)을 마치고, 아래 셀에서 3×3 figure를 구성하고 애니메이션을 저장합니다. 3×3 전체가 하나의 `fig`/`FuncAnimation`으로 묶여 있어, 3행3열을 포함한 모든 패널이 항상 하나의 통합된 출력으로 함께 나옵니다.)*

In [ ]:
# ---------- Figure 구성 (3x3) ----------
fig, axes = plt.subplots(3, 3, figsize=(19, 16), dpi=95, gridspec_kw={"wspace": 0.4, "hspace": 0.5})
ax_img_d4, ax_acc_d4, ax_realphase_d4 = axes[0, 0], axes[0, 1], axes[0, 2]
ax_temp_d4, ax_curv_d4, ax_phase_d4 = axes[1, 0], axes[1, 1], axes[1, 2]  # 1행3열 <-> 2행3열 교체
ax_pull_d4, ax_nugget_d4, ax_curve_d4 = axes[2, 0], axes[2, 1], axes[2, 2]  # 3행1열 <-> 3행2열 교체

# 1행1열 -- IR 이미지 + 축척바 + 컬러바
im_artist_d4 = ax_img_d4.imshow(frames_d4[0])
ax_img_d4.axis("off")
ax_img_d4.set_title("실측 IR 이미지", fontweight="bold")
label_text_d4 = ax_img_d4.text(0.03, 0.97, "", transform=ax_img_d4.transAxes, color="white",
                                fontsize=13, fontweight="bold", ha="left", va="top",
                                bbox=dict(boxstyle="round", facecolor="black", alpha=0.4, pad=0.5))
angle_text_d4 = ax_img_d4.text(0.97, 0.97, "", transform=ax_img_d4.transAxes, color="white",
                                fontsize=13, fontweight="bold", ha="right", va="top",
                                bbox=dict(boxstyle="round", facecolor="black", alpha=0.4, pad=0.5))

thermal_sm_d4 = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(vmin=0, vmax=1))
thermal_cbar_d4 = fig.colorbar(thermal_sm_d4, ax=ax_img_d4, fraction=0.046, pad=0.04)
thermal_cbar_d4.set_label("상대 열 강도 (Warmth)", fontsize=10, fontweight="bold")
thermal_cbar_d4.ax.tick_params(labelsize=8)
thermal_cbar_d4.set_ticks([0, 1])
thermal_cbar_d4.set_ticklabels(["낮음", "높음"])

if scalebar_px_10mm_d4 is not None:
    img_h_d4, img_w_d4 = frames_d4[0].shape[:2]
    y0_d4 = img_h_d4 - img_h_d4 * 0.06
    x0_d4 = (img_w_d4 - scalebar_px_10mm_d4) / 2.0
    x1_d4 = x0_d4 + scalebar_px_10mm_d4
    ax_img_d4.plot([x0_d4, x1_d4], [y0_d4, y0_d4], color="white", lw=3, solid_capstyle="butt")
    ax_img_d4.text((x0_d4 + x1_d4) / 2.0, y0_d4 - img_h_d4 * 0.04, "10mm",
                    color="white", fontsize=13, fontweight="bold", ha="center")

# 1행2열 -- 누적 정확도
ax_acc_d4.set_xlim(0, max(n_frames_d4 - 1, 1))
ax_acc_d4.set_ylim(0, 1.05)
ax_acc_d4.set_xlabel("샘플 순번"); ax_acc_d4.set_ylabel("누적 정확도")
ax_acc_d4.set_title("누적 정확도 (SIMULATED 참값 기준)\nbaseline vs adaptive", fontweight="bold")
ax_acc_d4.grid(True, ls=":", alpha=0.4)
baseline_line_d4, = ax_acc_d4.plot([], [], color="#7f7f7f", lw=2, label="baseline")
adaptive_line_d4, = ax_acc_d4.plot([], [], color="#2ca02c", lw=2, label="adaptive")
baseline_point_d4, = ax_acc_d4.plot([], [], marker="o", color="#7f7f7f", markersize=8, zorder=5)
adaptive_point_d4, = ax_acc_d4.plot([], [], marker="o", color="#2ca02c", markersize=8, zorder=5)
ax_acc_d4.legend(loc="lower right")

# 2행3열 -- 공정 위상도
ax_phase_d4.contourf(QQ_d4, TT_d4, combo_zone_d4, levels=[-0.5, 0.5, 1.5, 2.5], cmap=zone_cmap_d4)
# 1행3열(실측 위상도)처럼, 99건 전체의 참값(SIMULATED) 위치를 배경으로 미리 표시해 전체 분포를 한눈에 보여준다
ax_phase_d4.scatter(Q_true_arr, tilt_true_arr, s=14, color="#555555", alpha=0.35, zorder=2,
                     label="전체 99건(참값·SIMULATED)")
ax_phase_d4.set_xlabel("열입력 $Q$"); ax_phase_d4.set_ylabel("유효 접촉각 (도)")
ax_phase_d4.set_ylim(tilt_lo_d4, tilt_hi_d4)
Q_phase_all_d4 = np.concatenate([Q_true_arr, Q_base_arr, Q_adapt_arr])  # 이 패널에 실제로 표시되는 99건 기준으로 범위 산정
ax_phase_d4.set_xlim(0, Q_phase_all_d4.max() * 1.1)
ax_phase_d4.set_title("공정 위상도:\n참값(SIMULATED) vs baseline vs adaptive", fontweight="bold")
ax_phase_d4.grid(True, ls=":", color="gray", alpha=0.5, zorder=1.5)
ax_phase_d4.axhline(0, color="black", ls="--", lw=1, zorder=2)
zone_legend_d4 = [Patch(facecolor='#f8d7d7', edgecolor='gray', label='위험'),
                   Patch(facecolor='#ffe9a8', edgecolor='gray', label='경계'),
                   Patch(facecolor='#d5f2d5', edgecolor='gray', label='안전')]
zone_legend_artist_d4 = ax_phase_d4.legend(handles=zone_legend_d4, loc='upper right', fontsize=9, framealpha=0.9)
ax_phase_d4.add_artist(zone_legend_artist_d4)  # 아래 마커 범례를 추가해도 이 범례가 사라지지 않도록 유지
true_point_d4, = ax_phase_d4.plot([], [], marker="*", markersize=20, color="black",
                                    markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="참값(SIMULATED)")
base_point_d4, = ax_phase_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                    markeredgecolor="black", zorder=5, label="baseline")
adapt_point_d4, = ax_phase_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                     markeredgecolor="black", zorder=5, label="adaptive")
# 각 심벌(참값=SIMULATED/baseline/adaptive/전체 99건)이 무엇을 뜻하는지 좌측 위에 별도 범례로 설명
ax_phase_d4.legend(loc='upper left', fontsize=8, framealpha=0.9)

# 2행1열 -- 표면온도 편차
temp_pad = 3.0
ax_temp_d4.set_xlim(0, max(n_frames_d4 - 1, 1))
ax_temp_d4.set_ylim(-25, 65)

# 2행1열 배경: 이 샘플의 전류·통전시간(I, t) 기준으로, 온도편차(ΔT)가 얼마나 낮아야 Q가
# Q_min_train 아래로 떨어져 Bad 위험 구간에 들어가는지를 샘플별로 계산해 배경으로 표시한다
# (Q = I^2 * (1+ALPHA_R_PER_C*ΔT) * t 를 ΔT에 대해 역산).
temp_I_d4 = demo99_df["avg_current_A"].values
temp_t_d4 = demo99_df["weld_time_s"].values
temp_dT_lo_d4 = ((Q_min_train - Q_min_err_train) / (temp_I_d4 ** 2 * temp_t_d4) - 1.0) / ALPHA_R_PER_C
temp_dT_hi_d4 = ((Q_min_train + Q_min_err_train) / (temp_I_d4 ** 2 * temp_t_d4) - 1.0) / ALPHA_R_PER_C
x_temp_zone_d4 = np.arange(n_frames_d4)
temp_ylo_d4, temp_yhi_d4 = ax_temp_d4.get_ylim()
ax_temp_d4.fill_between(x_temp_zone_d4, temp_ylo_d4, temp_dT_lo_d4, color="#f8d7d7", zorder=0.5)
ax_temp_d4.fill_between(x_temp_zone_d4, temp_dT_lo_d4, temp_dT_hi_d4, color="#ffe9a8", zorder=0.5)
ax_temp_d4.fill_between(x_temp_zone_d4, temp_dT_hi_d4, temp_yhi_d4, color="#d5f2d5", zorder=0.5)
temp_zone_legend_d4 = [Patch(facecolor="#f8d7d7", edgecolor="gray", label="위험"),
                        Patch(facecolor="#ffe9a8", edgecolor="gray", label="경계"),
                        Patch(facecolor="#d5f2d5", edgecolor="gray", label="안전")]
temp_zone_legend_artist_d4 = ax_temp_d4.legend(handles=temp_zone_legend_d4, loc="lower right", fontsize=7, framealpha=0.9)
ax_temp_d4.add_artist(temp_zone_legend_artist_d4)

ax_temp_d4.set_xlabel("샘플 순번"); ax_temp_d4.set_ylabel("온도편차 (\u00b0C)")
ax_temp_d4.set_title("표면온도 편차:\n 실제(SIMULATED) vs 센서 감지값", fontweight="bold")
ax_temp_d4.grid(True, ls=":", alpha=0.4)
ax_temp_d4.axhline(0, color="black", ls="--", lw=1, zorder=2)
temp_true_line, = ax_temp_d4.plot([], [], color="#d62728", lw=2, label="실제(SIMULATED)")
temp_obs_line, = ax_temp_d4.plot([], [], color="#1f77b4", lw=1.5, alpha=0.7, label="센서 감지값")
temp_true_point, = ax_temp_d4.plot([], [], marker="o", color="#d62728", markersize=7, zorder=5)
temp_obs_point, = ax_temp_d4.plot([], [], marker="o", color="#1f77b4", markersize=7, zorder=5)
ax_temp_d4.legend(loc="upper left")

# 2행2열 -- 굴곡 편차
curv_pad = 2.0
ax_curv_d4.set_xlim(0, max(n_frames_d4 - 1, 1))
ax_curv_d4.set_ylim(tilt_lo_d4, tilt_hi_d4)  # 2행3열과 세로축 범위 통일

# 2행2열 배경: 이 샘플의 전극각도(angle_deg) 기준으로, 굴곡편차(Δtilt)가 얼마나 커야 유효
# 접촉각(angle_deg+Δtilt)이 EXPLODE_TILT_THRESHOLD_DEG±TILT_BAND_DEG를 넘는지 샘플별로 계산한다.
curv_angle_d4 = demo99_df["angle_deg"].values
curv_dtilt_lo_d4 = (EXPLODE_TILT_THRESHOLD_DEG - TILT_BAND_DEG) - curv_angle_d4  # 이보다 작으면 안전
curv_dtilt_hi_d4 = (EXPLODE_TILT_THRESHOLD_DEG + TILT_BAND_DEG) - curv_angle_d4  # 이보다 크면 위험
x_curv_zone_d4 = np.arange(n_frames_d4)
curv_ylo_d4, curv_yhi_d4 = ax_curv_d4.get_ylim()
ax_curv_d4.fill_between(x_curv_zone_d4, curv_ylo_d4, curv_dtilt_lo_d4, color="#d5f2d5", zorder=0.5)
ax_curv_d4.fill_between(x_curv_zone_d4, curv_dtilt_lo_d4, curv_dtilt_hi_d4, color="#ffe9a8", zorder=0.5)
ax_curv_d4.fill_between(x_curv_zone_d4, curv_dtilt_hi_d4, curv_yhi_d4, color="#f8d7d7", zorder=0.5)
curv_zone_legend_d4 = [Patch(facecolor="#f8d7d7", edgecolor="gray", label="위험"),
                        Patch(facecolor="#ffe9a8", edgecolor="gray", label="경계"),
                        Patch(facecolor="#d5f2d5", edgecolor="gray", label="안전")]
curv_zone_legend_artist_d4 = ax_curv_d4.legend(handles=curv_zone_legend_d4, loc="lower right", fontsize=7, framealpha=0.9)
ax_curv_d4.add_artist(curv_zone_legend_artist_d4)

ax_curv_d4.set_xlabel("샘플 순번"); ax_curv_d4.set_ylabel("굴곡편차 (도)")
ax_curv_d4.set_title("굴곡 편차:\n 실제(SIMULATED) vs 센서 감지값", fontweight="bold")
ax_curv_d4.grid(True, ls=":", alpha=0.4)
ax_curv_d4.axhline(0, color="black", ls="--", lw=1, zorder=2)
curv_true_line, = ax_curv_d4.plot([], [], color="#d62728", lw=2, label="실제(SIMULATED)")
curv_obs_line, = ax_curv_d4.plot([], [], color="#1f77b4", lw=1.5, alpha=0.7, label="센서 감지값")
curv_true_point, = ax_curv_d4.plot([], [], marker="o", color="#d62728", markersize=7, zorder=5)
curv_obs_point, = ax_curv_d4.plot([], [], marker="o", color="#1f77b4", markersize=7, zorder=5)
ax_curv_d4.legend(loc="upper left")

# 3행3열 -- Bad 확률 피팅 곡선
ax_curve_d4.errorbar(bad_binned_d4["heat_mean"], bad_binned_d4["bad_rate"], yerr=bad_binned_d4["bad_rate_sem"],
                      fmt="o", color="#1f77b4", ecolor="#d62728", capsize=4, markersize=6, zorder=3,
                      label="구간별 실측 (Mean ± SEM)")
ax_curve_d4.plot(Q_curve_d4, bad_curve_d4, color="#2ca02c", lw=2, label="피팅된 Bad 확률 곡선")
ax_curve_d4.axhline(0.5, color="gray", ls=":", lw=1)
ax_curve_d4.set_xlabel("열입력 $Q$"); ax_curve_d4.set_ylabel("Bad 확률(피팅값)")
ax_curve_d4.set_ylim(-0.05, 1.05)
ax_curve_d4.set_xlim(*Q_XLIM_D4)
ax_curve_d4.set_title("Bad 확률 피팅 함수:\n참값(SIMULATED) vs baseline vs adaptive", fontweight="bold")
ax_curve_d4.grid(True, ls=":", alpha=0.4)
true_curve_point, = ax_curve_d4.plot([], [], marker="*", markersize=20, color="black",
                                       markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="참값(SIMULATED)")
base_curve_point, = ax_curve_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                       markeredgecolor="black", zorder=5, label="baseline")
adapt_curve_point, = ax_curve_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                        markeredgecolor="black", zorder=5, label="adaptive")
ax_curve_d4.legend(loc="upper right", fontsize=9)

# 3행2열 -- 너겟 지름 성장 곡선 (1_rsw_optimization.ipynb D-2 피팅 재사용)
ax_nugget_d4.errorbar(bad_binned_d4["heat_mean"], bad_binned_d4["nugget_mean"], yerr=bad_binned_d4["nugget_sem"],
                       fmt="o", color="#1f77b4", ecolor="#d62728", capsize=4, markersize=6, zorder=3,
                       label="구간별 실측 (Mean ± SEM)")
ax_nugget_d4.plot(Q_curve_d4, nugget_curve_d4, color="#9467bd", lw=2, label="피팅된 너겟 성장 곡선")
ax_nugget_d4.set_xlabel("열입력 $Q$"); ax_nugget_d4.set_ylabel("예측 너겟지름 (mm)")
ax_nugget_d4.set_xlim(*Q_XLIM_D4)
ax_nugget_d4.set_title("너겟 지름 성장 곡선:\n참값(SIMULATED) vs baseline vs adaptive", fontweight="bold")
ax_nugget_d4.grid(True, ls=":", alpha=0.4)
true_nugget_point, = ax_nugget_d4.plot([], [], marker="*", markersize=20, color="black",
                                         markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="참값(SIMULATED)")
base_nugget_point, = ax_nugget_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                         markeredgecolor="black", zorder=5, label="baseline")
adapt_nugget_point, = ax_nugget_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                          markeredgecolor="black", zorder=5, label="adaptive")
ax_nugget_d4.legend(loc="lower right", fontsize=9)

# 3행1열 -- 인장강도 예측 (너겟지름-인장강도 선형회귀)
ax_pull_d4.errorbar(pull_binned_d4["nugget_mean"], pull_binned_d4["pull_mean"],
                     xerr=pull_binned_d4["nugget_sem"], yerr=pull_binned_d4["pull_sem"],
                     fmt="o", color="#1f77b4", ecolor="#d62728", capsize=4, markersize=6, zorder=3,
                     label="구간별 실측 (Mean ± SEM)")
ax_pull_d4.plot(nugget_range_d4, pull_curve_d4, color="#ff7f0e", lw=2,
                 label=f"선형 피팅 (r={pull_lr_d4.rvalue:.2f})")
ax_pull_d4.set_xlabel("예측 너겟지름 (mm)"); ax_pull_d4.set_ylabel("예측 인장강도 (N)")
ax_pull_d4.set_title("인장강도 예측:\n참값(SIMULATED) vs baseline vs adaptive", fontweight="bold")
ax_pull_d4.grid(True, ls=":", alpha=0.4)
true_pull_point, = ax_pull_d4.plot([], [], marker="*", markersize=20, color="black",
                                     markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="참값(SIMULATED)")
base_pull_point, = ax_pull_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                     markeredgecolor="black", zorder=5, label="baseline")
adapt_pull_point, = ax_pull_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                      markeredgecolor="black", zorder=5, label="adaptive")
ax_pull_d4.legend(loc="lower right", fontsize=9)

# 1행3열 -- 공정 위상도(전류 x 통전시간): 1_rsw_optimization.ipynb 'RSW 실시간 종합 모니터링
# 대시보드'의 1행2열을 그대로 이식한다. 이 노트북 자체의 Q x 유효접촉각 위상도(2행3열)와는 달리,
# 원래 물리량(전류/통전시간) 축 위에서 전극각도별 배경을 매 프레임 다시 그린다.
PHASE_I_MAX_D4 = 5000.0
PHASE_T_MAX_D4 = 2.0
I_grid_d4 = np.linspace(0.0, PHASE_I_MAX_D4, 220)
t_grid_d4 = np.linspace(0.0, PHASE_T_MAX_D4, 220)
II_real_d4, TT_time_d4 = np.meshgrid(I_grid_d4, t_grid_d4)
QQ_real_d4 = II_real_d4 ** 2 * TT_time_d4  # heat_input_proxy와 동일한 정의(R=1 가정)
zone_real_d4 = np.where(QQ_real_d4 < Q_min_train - Q_min_err_train, 0,
                         np.where(QQ_real_d4 < Q_min_train + Q_min_err_train, 1, 2))
cat_colors_d4 = {'Good': '#2ca02c', 'Bad': '#d62728', 'Explode': '#ff7f0e'}


def draw_realphase_bg_d4(ax, angle):
    ax.clear()
    ax.contourf(II_real_d4, TT_time_d4, zone_real_d4, levels=[-0.5, 0.5, 1.5, 2.5], cmap=zone_cmap_d4)
    cs = ax.contour(II_real_d4, TT_time_d4, QQ_real_d4, levels=8, colors='gray', linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=8, fmt='%.0e')
    ax.contour(II_real_d4, TT_time_d4, QQ_real_d4, levels=[Q_min_train - Q_min_err_train, Q_min_train + Q_min_err_train], colors='#b8860b', linestyles='--', linewidths=1.2)  # 경계 폭이 좁아 채움만으로는 잘 안 보이므로 점선으로 위치를 명시
    for cat, color in cat_colors_d4.items():
        s = agg[(agg["category"] == cat) & (agg["angle_deg"] == angle)]
        ax.scatter(s["avg_current_A"], s["weld_time_s"], s=12, color=color, alpha=0.3)
    ax.set_xlabel("전류 Current (A)"); ax.set_ylabel("통전시간 Weld Time (s)")
    ax.set_xlim(0, PHASE_I_MAX_D4); ax.set_ylim(0, PHASE_T_MAX_D4)
    ax.grid(True, ls=":", color="gray", alpha=0.5, zorder=1.5)
    zone_legend_real_d4 = [Patch(facecolor='#f8d7d7', edgecolor='gray', label='위험'),
                            Patch(facecolor='#ffe9a8', edgecolor='gray', label='경계'),
                            Patch(facecolor='#d5f2d5', edgecolor='gray', label='안전')]
    ax.legend(handles=zone_legend_real_d4, loc='upper right', fontsize=8, framealpha=0.9)

fig.suptitle("RSW 적응형 파인튜닝 실시간 종합 모니터링 대시보드 (IR 이미지 보유 99건, 시연용)",
             fontsize=17, fontweight="bold")


def update_d4(i):
    row = demo99_df.iloc[i]
    im_artist_d4.set_data(frames_d4[i])
    label_text_d4.set_text(f"Sample #{int(row['sample_id'])}")
    angle_text_d4.set_text(f"전극각도 {row['angle_deg']:g}\u00b0")

    baseline_line_d4.set_data(np.arange(i + 1), baseline_rolling_acc_d4[:i + 1])
    adaptive_line_d4.set_data(np.arange(i + 1), adaptive_rolling_acc_d4[:i + 1])
    baseline_point_d4.set_data([i], [baseline_rolling_acc_d4[i]])
    adaptive_point_d4.set_data([i], [adaptive_rolling_acc_d4[i]])

    true_point_d4.set_data([Q_true_arr[i]], [tilt_true_arr[i]])
    base_point_d4.set_data([Q_base_arr[i]], [tilt_base_arr[i]])
    adapt_point_d4.set_data([Q_adapt_arr[i]], [tilt_adapt_arr[i]])

    temp_true_line.set_data(np.arange(i + 1), temp_true_arr[:i + 1])
    temp_obs_line.set_data(np.arange(i + 1), temp_obs_arr[:i + 1])
    temp_true_point.set_data([i], [temp_true_arr[i]])
    temp_obs_point.set_data([i], [temp_obs_arr[i]])

    curv_true_line.set_data(np.arange(i + 1), curv_true_arr[:i + 1])
    curv_obs_line.set_data(np.arange(i + 1), curv_obs_arr[:i + 1])
    curv_true_point.set_data([i], [curv_true_arr[i]])
    curv_obs_point.set_data([i], [curv_obs_arr[i]])

    true_curve_point.set_data([Q_true_arr[i]], [bad_rate_logistic_model(Q_true_arr[i], Q_min_train, k_train)])
    base_curve_point.set_data([Q_base_arr[i]], [bad_rate_logistic_model(Q_base_arr[i], Q_min_train, k_train)])
    adapt_curve_point.set_data([Q_adapt_arr[i]], [bad_rate_logistic_model(Q_adapt_arr[i], Q_min_train, k_train)])

    true_nugget_point.set_data([Q_true_arr[i]], [nugget_true_arr[i]])
    base_nugget_point.set_data([Q_base_arr[i]], [nugget_base_arr[i]])
    adapt_nugget_point.set_data([Q_adapt_arr[i]], [nugget_adapt_arr[i]])

    true_pull_point.set_data([nugget_true_arr[i]], [pull_true_arr[i]])
    base_pull_point.set_data([nugget_base_arr[i]], [pull_base_arr[i]])
    adapt_pull_point.set_data([nugget_adapt_arr[i]], [pull_adapt_arr[i]])

    # 1행3열 (draw_realphase_bg_d4가 ax.clear()를 하므로 점·텍스트도 매번 다시 그림)
    draw_realphase_bg_d4(ax_realphase_d4, row["angle_deg"])
    ax_realphase_d4.scatter([row["avg_current_A"]], [row["weld_time_s"]], s=400, marker="*",
                             color="black", edgecolor="yellow", linewidth=1.5, zorder=5)
    explode_tag_d4 = "안전" if row["effective_tilt_true_deg"] <= EXPLODE_TILT_THRESHOLD_DEG else "위험 \u2191"  # angle_deg만이 아니라 굴곡편차까지 반영한 유효 접촉각 기준
    ax_realphase_d4.set_title(f"공정 위상도(실측): \n전극각도 {row['angle_deg']:g}\u00b0 (Explode {explode_tag_d4})",
                               fontweight="bold")
    ax_realphase_d4.text(
        0.98, 0.02,
        f"전류 {row['avg_current_A']:.0f}A / 시간 {row['weld_time_s']:.2f}s\n"
        f"압력 {row['pressure_psi']:.0f}PSI / 각도 {row['angle_deg']:g}\u00b0",
        transform=ax_realphase_d4.transAxes, fontsize=10, fontweight="bold", va="bottom", ha="right",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, pad=0.5))

    return (im_artist_d4, label_text_d4, angle_text_d4, baseline_line_d4, adaptive_line_d4,
            baseline_point_d4, adaptive_point_d4, true_point_d4, base_point_d4, adapt_point_d4,
            temp_true_line, temp_obs_line, temp_true_point, temp_obs_point,
            curv_true_line, curv_obs_line, curv_true_point, curv_obs_point,
            true_curve_point, base_curve_point, adapt_curve_point,
            true_nugget_point, base_nugget_point, adapt_nugget_point,
            true_pull_point, base_pull_point, adapt_pull_point)


ANIM_INTERVAL_MS_D4 = 250
ANIM_FPS_D4 = 4
anim_d4 = FuncAnimation(fig, update_d4, frames=n_frames_d4, interval=ANIM_INTERVAL_MS_D4, blit=False)

gif_path_d4 = os.path.join(FIGURE_DIR, "adaptive_dashboard.gif")
anim_d4.save(gif_path_d4, writer=PillowWriter(fps=ANIM_FPS_D4))
print(f"[완료] {n_frames_d4}프레임 3x3 종합 대시보드 저장(GIF): {gif_path_d4}")

plt.rcParams["animation.embed_limit"] = 80
from IPython.display import HTML, display as ipy_display
interactive_player_d4 = HTML(anim_d4.to_jshtml())
plt.close(fig)
ipy_display(interactive_player_d4)
print(f"[최종] baseline 누적 정확도 {baseline_rolling_acc_d4[-1] * 100:.1f}%, "
      f"adaptive 누적 정확도 {adaptive_rolling_acc_d4[-1] * 100:.1f}%")

**패널별 데이터 출처.** 이 대시보드는 D-1/D-2 공식 TEST 지표(50건)와는 다른 표본 — IR 이미지가 있는 99건 전체(train/valid/test 혼합) — 을 사용하는 시연용입니다. 이미 확정된 표준값 모델과 VALID 보정계수를 고정된 시스템으로 보고 새 입력에 적용해 볼 뿐, 이 단계에서 추가 학습은 없습니다.

| 위치 | 내용 | 근거 |
|---|---|---|
| 1행1열 | 실측 IR(적외선 열화상) 이미지 + 10mm 축척바 + 상대 열강도 컬러바 + 전극각도 표시 | 원본 사진(현재 프레임, D-3 참고) |
| 1행2열 | baseline vs adaptive 누적(rolling) 정확도 | B-8 예측 결과 |
| 1행3열 | 실측 공정 위상도(전류×통전시간), 전극각도별 배경 재구성 | agg 실측값 |
| 2행1열 | 표면온도 편차: 실제(SIMULATED) vs 센서 감지값 + 샘플별 Bad 위험 경계 배경 | B-5/B-6, Q_min_train 역산 |
| 2행2열 | 굴곡 편차: 실제(SIMULATED) vs 센서 감지값 + 샘플별 Explode 위험 경계 배경 | B-5/B-6, EXPLODE_TILT_THRESHOLD_DEG 역산 |
| 2행3열 | 공정 위상도($Q$ × 유효 접촉각), 99건 배경 산점 + 참값(SIMULATED)·baseline·adaptive 3점, y=0 기준선 | B-3/B-4 표준값 모델 |
| 3행1열 | 인장강도 예측 + 구간별 실측 평균±SEM 에러바 | agg 실측값 qcut 구간화 |
| 3행2열 | 너겟 지름 성장 곡선(가로축 0~2.0e7) + 구간별 실측 평균±SEM 에러바 | `step4_nugget_fit_params.csv`, `step2_binned_stats.csv` |
| 3행3열 | Bad 확률 피팅 곡선(가로축 0~2.0e7) + 구간별 실측 평균±SEM 에러바 | B-3, `step2_binned_stats.csv` |

3행2열·3행3열의 에러바는 "그럴듯하게 그어진 선"이 아니라 실제 관측 데이터에 피팅된 결과임을 뒷받침합니다. 1행3열·2행3열은 서로 다른 두 축 공간(합성 편차가 반영된 유효값 vs 원래 실측 물리량)에서 같은 판정 문제를 바라보는 두 관점을 나란히 제공합니다.

### D-5 — 센서 보정 학습 이력 시각화 (Epoch)

B-7에서 경사하강법으로 온도·굴곡 센서 보정계수를 학습하는 과정을 epoch별로 추적한 결과입니다.

| 위치 | 내용 |
|---|---|
| 1행1열 | Train Loss (calib_train, MSE) |
| 1행2열 | F1-score / Accuracy (calib_valid) |
| 2행1열 | Valid Loss (calib_valid, MSE) |
| 2행2열 | Learning Rate (epoch에 따라 감쇠) |

이 학습은 VALID(약 98건) 내부의 calib_train/calib_valid에서만 이뤄지며, TEST는 이 단계에서 전혀
사용하지 않습니다.

**이 학습 이력은 전부 합성(SIMULATED) 데이터 기반입니다.** 손실을 계산하는 목표값(`sim_temp_dev_C`,
`sim_curvature_dev_deg`)과 입력값(`obs_temp_dev_C`, `obs_curvature_dev_deg`)이 모두 B-5/B-6에서
생성한 합성 편차·합성 센서 관측값이며, 실제 로봇이나 센서로 측정한 값이 아닙니다.

**Train Loss와 Valid Loss가 거의 같게 보이는 것은 정상입니다.** calib_train과 calib_valid는 같은
분포(B-5의 동일한 `rng.uniform`/`rng.normal` 생성 규칙)에서 무작위로 나눈 것일 뿐 서로 다른 조건을
대표하지 않으며, 학습하는 모델도 선형회귀 파라미터 2개(`a`, `b`)뿐인 매우 단순한 모델이라 약 68건의
calib_train만으로도 과적합 여지가 거의 없습니다. 따라서 두 손실이 비슷하게 수렴하는 것은 버그가
아니라 "이 데이터·모델 규모에서는 과적합이 나타나지 않는다"는 것을 보여주는 정상적인 결과입니다.

In [ ]:
# ==========================================
# [D-5] 센서 보정 학습 이력 시각화 (Epoch별 Train/Valid Loss, F1&Accuracy, LR)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(12, 9), dpi=105)
ax_tl, ax_f1 = axes[0, 0], axes[0, 1]
ax_vl, ax_lr = axes[1, 0], axes[1, 1]

ax_tl.plot(epoch_history_df["epoch"], epoch_history_df["train_loss_temp"], color="#d62728", lw=2, marker="o", markersize=4, label="온도센서")
ax_tl.plot(epoch_history_df["epoch"], epoch_history_df["train_loss_curv"], color="#1f77b4", lw=2, marker="o", markersize=4, label="굴곡센서")
ax_tl.set_xlabel("Epoch"); ax_tl.set_ylabel("Train Loss (MSE)")
ax_tl.set_title("Train Loss", fontweight="bold")
ax_tl.grid(True, ls=":", alpha=0.4); ax_tl.legend()

ax_f1.plot(epoch_history_df["epoch"], epoch_history_df["valid_f1"], color="#2ca02c", lw=2, marker="o", markersize=4, label="F1-score")
ax_f1.plot(epoch_history_df["epoch"], epoch_history_df["valid_accuracy"], color="#9467bd", lw=2, marker="o", markersize=4, label="Accuracy")
ax_f1.set_xlabel("Epoch"); ax_f1.set_ylabel("지표 값")
ax_f1.set_ylim(0, 1.05)
ax_f1.set_title("F1-score / Accuracy (calib_valid)", fontweight="bold")
ax_f1.grid(True, ls=":", alpha=0.4); ax_f1.legend()

ax_vl.plot(epoch_history_df["epoch"], epoch_history_df["valid_loss_temp"], color="#d62728", lw=2, marker="o", markersize=4, label="온도센서")
ax_vl.plot(epoch_history_df["epoch"], epoch_history_df["valid_loss_curv"], color="#1f77b4", lw=2, marker="o", markersize=4, label="굴곡센서")
ax_vl.set_xlabel("Epoch"); ax_vl.set_ylabel("Valid Loss (MSE)")
ax_vl.set_title("Valid Loss", fontweight="bold")
ax_vl.grid(True, ls=":", alpha=0.4); ax_vl.legend()

ax_lr.plot(epoch_history_df["epoch"], epoch_history_df["lr"], color="#ff7f0e", lw=2, marker="o", markersize=4)
ax_lr.set_xlabel("Epoch"); ax_lr.set_ylabel("Learning Rate")
ax_lr.set_title("Learning Rate", fontweight="bold")
ax_lr.grid(True, ls=":", alpha=0.4)

fig.suptitle("센서 보정 파인튜닝 학습 이력 (VALID 내부 calib_train/calib_valid)",
             fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "calibration_training_history.png"))
plt.show()

**결과 해석(미니배치 SGD 재실행 결과).** `result_rsw_adaptive/step4_sensor_calibration.csv`에 저장된 60 epoch 학습 후 최종 보정계수는 온도센서 `corrected = 0.967 x obs + 0.156`, 굴곡센서 `corrected = 0.925 x obs + 0.749`입니다. 두 계수 모두 기울기가 1에 가까워, 학습된 보정이 센서 관측값을 거의 그대로 신뢰하는 방향으로 수렴했음을 보여줍니다(관측 노이즈가 평균 0인 가우시안이었으므로 합리적인 결과입니다).

`step4_sensor_calibration_history.csv` 기준 마지막 epoch(60)에서 calib_valid F1=0.979, accuracy=0.967이고, train loss는 온도 10.07 / 굴곡 2.21, valid loss는 온도 5.84 / 굴곡 2.09입니다. 미니배치 SGD로 바꾼 뒤로는 F1·accuracy가 학습 초반 0.868/0.767에서 후반 0.979/0.967까지 실제로 변화하는 모습을 보여, 이전(전체배치, F1·accuracy가 처음부터 1.0으로 고정되어 있던)보다 학습 진행 과정을 더 현실적으로 보여줍니다.

**Train Loss와 Valid Loss가 비슷한 수준인 것은 정상입니다** — calib_train과 calib_valid는 동일한 분포(B-5의 동일 `rng.uniform`/`rng.normal` 규칙)에서 무작위로 나눈 표본일 뿐이고, 학습 대상도 선형 파라미터 2개(`a`, `b`)뿐인 매우 단순한 모델이라 과적합 여지가 거의 없기 때문입니다(F-2에서 다시 다루는 정직성 표기와 일관됩니다). valid loss가 train loss보다 오히려 낮게 나온 것도 오류가 아니라, calib_valid(~29건)가 작은 표본이라 우연히 적합선에 더 가까운 샘플들이 뽑혔을 가능성을 보여주는 정상적인 표본 변동입니다. 이 학습 이력 전체가 `sim_`/`obs_` 접두사가 붙은 합성 데이터에 기반한다는 점도 함께 유의해야 합니다.

## E. 결과 정리 및 분석

### E-1 — 최종 종합 해석

#### 1. 무엇을 검증했는가

표준값(train에서 학습한 물리모델)을 고정 적용하는 baseline과, 합성 굴곡·표면온도 편차를 센서로
감지하고 VALID로 보정해 대응하는 adaptive를 TEST(약 50건)에서 비교했습니다. Bad(열입력 부족)와
Explode(굴곡) 두 메커니즘을 각각 독립적으로 평가했습니다.

#### 2. 무엇이 확인되었는가

D-1의 accuracy·precision·recall·F1 비교와 D-2의 혼동행렬을 종합하면, **Explode(굴곡) 과제**에서는
adaptive가 baseline보다 뚜렷이 우수합니다(accuracy 0.76→0.96, recall 0.71→0.98). 반면 **Bad(열입력)
과제**는 이번 TEST 표본(양성 2건뿐)에서 baseline·adaptive 지표가 완전히 동일(둘 다 accuracy 1.00)해,
이 지표만으로 개선 여부를 판단할 근거가 부족합니다(구체적 수치는 `step6_baseline_vs_adaptive_metrics.csv`
참고). 이는 adaptive가 무의미하다는 뜻이 아니라 표본이 너무 작아 이 지표만으로는 검증할 수 없다는
뜻입니다. 종합하면, 편차가 존재하는 상황에서 그것을 무시하는 baseline이 최소한 Explode에서는
체계적으로 더 많이 틀리고, 불완전한 정보(노이즈가 섞인 센서)라도 활용하는 adaptive가 참값(SIMULATED)에
더 가깝게 수렴한다는 것을 확인했습니다.

#### 3. 이 결과를 어떻게 해석해야 하는가

**이 결과는 실측 미지 데이터에 대한 성능 검증이 아닙니다.** 참값 자체가 합성 편차로부터 계산된
시뮬레이션이므로, adaptive가 baseline을 능가하는 정도는 애초에 얼마나 큰 편차를 주입했는지와 센서
노이즈 크기를 얼마로 가정했는지에 좌우됩니다. 이 노트북이 실제로 보여주는 것은 편차를 감지하고
보정하는 메커니즘 자체가 원리적으로 작동한다는 방법론적 증명이며, 실제 로봇 용접 현장에 이 수치를
그대로 적용할 수 있다는 뜻이 아닙니다. 자세한 한계는 F-2에서 다룹니다.

In [ ]:
# ==========================================
# [E-1] 최종 요약 저장
# ==========================================
summary_lines = []
summary_lines.append("=" * 60)
summary_lines.append("  RSW 적응형 파인튜닝 -- 최종 요약 (시뮬레이션 기반)")
summary_lines.append("=" * 60)
summary_lines.append(f"표준값(train, 실측): Q_min={Q_min_train:.3e}, k={k_train:.3e}, "
                      f"Explode 임계={EXPLODE_TILT_THRESHOLD_DEG}도")
summary_lines.append("")
summary_lines.append(metrics_df.to_string(index=False))
summary_lines.append("")
summary_lines.append("주의: true_* 컬럼은 실측 결과가 아니라 물리모델 기반 시뮬레이션이다.")

summary_text = "\n".join(summary_lines)
print(summary_text)
with open(os.path.join(RESULT_DIR, "step7_final_summary.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)


## F. 트러블슈팅 및 개선 방안

### F-1 — 발견한 이슈 및 설계 판단

| # | 이슈 | 판단/해결 |
|---|---|---|
| 1 | 사용자가 원한 9개 변수(전류·전압·속도·압력·시간·깊이·굴곡·온도·습도)를 모두 실측한 공개 데이터셋 부재 | RSW 실측(전류·시간·압력·각도) + 굴곡·표면온도만 합성하는 하이브리드로 축소(습도는 RSW와 물리적 연관이 약해 제외) |
| 2 | 습도 포함 여부 | RSW는 밀폐 접촉 공정이라 GMAW 같은 개방 아크 공정과 달리 습도의 직접적 물리 경로가 약함 -- 제외 |
| 3 | adaptive가 참값과 동일한 정보를 그대로 쓰면 비교가 무의미(동어반복) | 센서 관측에 가우시안 노이즈를 추가하고, VALID로만 보정계수를 학습해 TEST에 적용(data leakage 방지)함으로써 adaptive도 불완전한 정보로 판단하게 설계 |
| 4 | Bad와 Explode를 하나의 지표로 합치면 서로 다른 결함 메커니즘이 뒤섞임 | 원 노트북의 원칙(두 결함을 항상 별도로 다룸)을 그대로 유지해 두 과제를 독립적으로 비교 |

### F-2 — 구조적 한계 (반드시 읽어야 할 정직성 표기)

A-3에서 실측값과 합성값을 구분한 표를 먼저 제시했습니다. 여기서는 그 구분이 결과 해석에 갖는
함의를 상세히 설명합니다. **이 노트북을 해석할 때 아래 네 가지를 반드시 함께 고려해야 합니다.**

1. **굴곡·표면온도 편차(`sim_temp_dev_C`, `sim_curvature_dev_deg`)는 실측이 아니라 물리적으로
   동기부여된 합성값입니다.** 온도가 저항을 바꾸고 굴곡이 접촉각을 바꾼다는 방향성은 물리적으로
   타당하지만, 그 크기와 분포(균등분포 [-15,45], [0,20])는 임의로 정한 가정값이지 이 데이터셋에서
   측정되거나 다른 문헌에서 인용한 값이 아닙니다. 특히 `sim_curvature_dev_deg`는 균등분포
   [0,20]으로 **항상 0 이상만 나오도록 설계**되어 있어 편차가 언제나 위험(Explode) 방향으로만
   작용합니다. 그 결과 TEST·VALID의 `true_explode_risk` 양성 비율이 원본 실측 Explode
   비율(6.3%)보다 훨씬 높게 나타나며, baseline 대비 adaptive의 Explode 개선 폭(D-1/D-2, E-1)이
   이 구조적 편향 때문에 실제보다 커 보일 수 있다는 점을 감안해야 합니다.
2. **VALID·TEST의 참값(ground truth)은 실제 관측된 결과가 아니라, 물리모델 기반 가상
   시나리오(counterfactual simulation)입니다.** 이 실측 용접에 이런 편차가 추가로 있었다면 이라는
   가정 위에서 계산된 값이며, 실제로 그 조건에서 용접이 이뤄진 적은 없습니다.
3. **`ALPHA_R_PER_C=0.004`와 `EXPLODE_TILT_THRESHOLD_DEG=7.5`는 근사·단순화된 가정이지, 데이터로
   피팅되거나 측정된 값이 아닙니다.** 전자는 철강 계열의 전형적인 저항 온도계수를 참고한 근사치이고,
   후자는 실측 두 수준(0도, 15도)의 단순 중간값입니다.
4. **이 노트북의 성능 비교는 실제 미지 데이터에 대한 예측 성능 검증이 아니라 방법론 시연입니다.**
   adaptive가 baseline을 능가하는 폭은 주입한 합성 편차의 크기, 가정한 센서 노이즈 수준에 따라
   달라질 수 있습니다. 이 노트북이 실제로 입증하는 것은 편차를 감지하고 보정하면 원리적으로 더 나은
   판정을 내릴 수 있다는 메커니즘의 타당성이지, 특정 수치(예: F1이 몇 퍼센트 개선되었는지)가 실제
   현장에서도 그대로 재현된다는 뜻이 아닙니다.

추가로, `EXPLODE_TILT_THRESHOLD_DEG`가 고정 규칙이라 adaptive조차 이 임계값 자체를 데이터로부터
다시 학습하지는 않는다는 점, 그리고 감지·대응 지연(latency) 없이 매 샘플이 즉시 보정된다고 가정한
점도 실제 로봇 제어 루프보다 단순화된 부분입니다. 또한 전류·전압·속도·압력·시간·깊이·굴곡·온도·습도를
모두 실측한 공개 데이터셋을 조사했으나 존재하지 않았고(온도·습도·전류·전압·결함라벨을 모두 갖춘
유일한 후보는 비공개 기업 데이터였음), 그런 경위로 이 하이브리드 방식을 채택했다는 점도 함께
밝힙니다.

### F-3 — 향후 개선 과제

1. 실제 굴곡·표면온도 센서 데이터를 확보할 수 있다면(예: 비전 기반 심 추적 센서, 접촉식 열전대),
   합성값을 실측으로 교체해 재검증한다.
2. `EXPLODE_TILT_THRESHOLD_DEG`처럼 고정된 임계 규칙 대신, 온라인으로 임계값 자체를 갱신하는 방식
   (`1_rsw_optimization.ipynb` B-8의 Phase 1 온라인 학습 시뮬레이션과 유사한 접근)을 도입한다.
3. 센서 노이즈 크기(3°C, 1.5도)에 대한 민감도 분석을 추가해, 노이즈가 커질수록 adaptive의 이점이
   어떻게 줄어드는지 정량화한다. **[완료] `B-7b`에서 실행 완료 -- Explode F1 격차가 무노이즈
   0.159에서 5배 노이즈에서 0.052까지 줄어들어 가설이 확인되었다(Bad는 격차 0으로 무관함).**
4. 온도-저항 관계를 선형 근사가 아니라 재질별 실측 계수로 교체한다.
5. 굴곡과 온도 편차의 상호작용(예: 굴곡이 클수록 접촉저항 변화가 더 민감해지는 효과)을 모델링한다.
6. Bad·Explode 지표(precision/recall/F1)에 부트스트랩 신뢰구간을 추가해, 소표본(TEST 50건, Bad 양성 2건)에서 지표가 얼마나 불안정한지 정량적으로 보여준다. **[완료] `D-1b`에서 실행 완료 -- Bad는 신뢰구간이 [0,1]로 사실상 무의미했고, Explode는 baseline·adaptive 신뢰구간이 겹치지 않아 개선이 통계적으로도 뒷받침되었다.**

### F-4 — 결론적으로

이 노트북은 실측 RSW 데이터(전류·통전시간·가압력·전극각도)를 표준값의 기반으로 삼고, 공개
데이터셋으로는 확보할 수 없었던 굴곡·표면온도 편차를 물리적으로 타당하게 합성하여, 표준값을 고정
적용하는 시스템과 편차를 감지하고 보정하는 적응형 시스템을 비교하는 방법론을 시연했습니다. 두
결함 메커니즘(Bad, Explode) 모두에서 적응형 접근이 더 나은 판정 지표를 보였으나, 이는 실제 현장
성능 보증이 아니라 센서 기반 실시간 대응이라는 아이디어가 원리적으로 타당함을 보여주는 시뮬레이션
증거로 한정해 해석해야 합니다. 실제 적용을 위해서는 F-2의 한계를 해소하는 실측 데이터 확보가
선행되어야 합니다.

In [ ]:
# ---------- Figure Configuration (3x3) eng. version ----------
fig, axes = plt.subplots(3, 3, figsize=(19, 16), dpi=95, gridspec_kw={"wspace": 0.4, "hspace": 0.5})
ax_img_d4, ax_acc_d4, ax_realphase_d4 = axes[0, 0], axes[0, 1], axes[0, 2]
ax_temp_d4, ax_curv_d4, ax_phase_d4 = axes[1, 0], axes[1, 1], axes[1, 2]  # Swap Row 1 Col 3 <-> Row 2 Col 3
ax_pull_d4, ax_nugget_d4, ax_curve_d4 = axes[2, 0], axes[2, 1], axes[2, 2]  # Swap Row 3 Col 1 <-> Row 3 Col 2

# Row 1 Col 1 -- IR Image + Scale Bar + Colorbar
im_artist_d4 = ax_img_d4.imshow(frames_d4[0])
ax_img_d4.axis("off")
ax_img_d4.set_title("Empirical IR Thermal Image", fontweight="bold")
label_text_d4 = ax_img_d4.text(0.03, 0.97, "", transform=ax_img_d4.transAxes, color="white",
                                fontsize=13, fontweight="bold", ha="left", va="top",
                                bbox=dict(boxstyle="round", facecolor="black", alpha=0.4, pad=0.5))
angle_text_d4 = ax_img_d4.text(0.97, 0.97, "", transform=ax_img_d4.transAxes, color="white",
                                fontsize=13, fontweight="bold", ha="right", va="top",
                                bbox=dict(boxstyle="round", facecolor="black", alpha=0.4, pad=0.5))

thermal_sm_d4 = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(vmin=0, vmax=1))
thermal_cbar_d4 = fig.colorbar(thermal_sm_d4, ax=ax_img_d4, fraction=0.046, pad=0.04)
thermal_cbar_d4.set_label("Relative Heat Intensity", fontsize=10, fontweight="bold")
thermal_cbar_d4.ax.tick_params(labelsize=8)
thermal_cbar_d4.set_ticks([0, 1])
thermal_cbar_d4.set_ticklabels(["Low", "High"])

if scalebar_px_10mm_d4 is not None:
    img_h_d4, img_w_d4 = frames_d4[0].shape[:2]
    y0_d4 = img_h_d4 - img_h_d4 * 0.06
    x0_d4 = (img_w_d4 - scalebar_px_10mm_d4) / 2.0
    x1_d4 = x0_d4 + scalebar_px_10mm_d4
    ax_img_d4.plot([x0_d4, x1_d4], [y0_d4, y0_d4], color="white", lw=3, solid_capstyle="butt")
    ax_img_d4.text((x0_d4 + x1_d4) / 2.0, y0_d4 - img_h_d4 * 0.04, "10mm",
                    color="white", fontsize=13, fontweight="bold", ha="center")

# Row 1 Col 2 -- Cumulative Accuracy
ax_acc_d4.set_xlim(0, max(n_frames_d4 - 1, 1))
ax_acc_d4.set_ylim(0, 1.05)
ax_acc_d4.set_xlabel("Sample Sequence"); ax_acc_d4.set_ylabel("Cumulative Accuracy")
ax_acc_d4.set_title("Cumulative Accuracy (vs SIMULATED Truth)\nbaseline vs adaptive", fontweight="bold")
ax_acc_d4.grid(True, ls=":", alpha=0.4)
baseline_line_d4, = ax_acc_d4.plot([], [], color="#7f7f7f", lw=2, label="baseline")
adaptive_line_d4, = ax_acc_d4.plot([], [], color="#2ca02c", lw=2, label="adaptive")
baseline_point_d4, = ax_acc_d4.plot([], [], marker="o", color="#7f7f7f", markersize=8, zorder=5)
adaptive_point_d4, = ax_acc_d4.plot([], [], marker="o", color="#2ca02c", markersize=8, zorder=5)
ax_acc_d4.legend(loc="lower right")

# Row 2 Col 3 -- Process Phase Diagram
ax_phase_d4.contourf(QQ_d4, TT_d4, combo_zone_d4, levels=[-0.5, 0.5, 1.5, 2.5], cmap=zone_cmap_d4)
# Display all 99 SIMULATED truth points in the background to show overall distribution
ax_phase_d4.scatter(Q_true_arr, tilt_true_arr, s=14, color="#555555", alpha=0.35, zorder=2,
                     label="All 99 Cases (SIMULATED Truth)")
ax_phase_d4.set_xlabel("Heat Input $Q$"); ax_phase_d4.set_ylabel("Effective Angle (deg)")
ax_phase_d4.set_ylim(tilt_lo_d4, tilt_hi_d4)
Q_phase_all_d4 = np.concatenate([Q_true_arr, Q_base_arr, Q_adapt_arr])  # Calculate range based on cases actually displayed
ax_phase_d4.set_xlim(0, Q_phase_all_d4.max() * 1.1)
ax_phase_d4.set_title("Process Phase Diagram:\nSIMULATED Truth vs baseline vs adaptive", fontweight="bold")
ax_phase_d4.grid(True, ls=":", color="gray", alpha=0.5, zorder=1.5)
ax_phase_d4.axhline(0, color="black", ls="--", lw=1, zorder=2)
zone_legend_d4 = [Patch(facecolor='#f8d7d7', edgecolor='gray', label='Risk'),
                   Patch(facecolor='#ffe9a8', edgecolor='gray', label='Caution'),
                   Patch(facecolor='#d5f2d5', edgecolor='gray', label='Safe')]
zone_legend_artist_d4 = ax_phase_d4.legend(handles=zone_legend_d4, loc='upper right', fontsize=9, framealpha=0.9)
ax_phase_d4.add_artist(zone_legend_artist_d4)  # Keep this legend from disappearing when adding the marker legend below
true_point_d4, = ax_phase_d4.plot([], [], marker="*", markersize=20, color="black",
                                    markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="SIMULATED Truth")
base_point_d4, = ax_phase_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                    markeredgecolor="black", zorder=5, label="baseline")
adapt_point_d4, = ax_phase_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                     markeredgecolor="black", zorder=5, label="adaptive")
# Separate legend on top left explaining symbols
ax_phase_d4.legend(loc='upper left', fontsize=8, framealpha=0.9)

# Row 2 Col 1 -- Surface Temp Deviation
temp_pad = 3.0
ax_temp_d4.set_xlim(0, max(n_frames_d4 - 1, 1))
ax_temp_d4.set_ylim(-25, 65)

# Row 2 Col 1 Background: Calculate and display required delta T based on current/time to indicate Risk zones
temp_I_d4 = demo99_df["avg_current_A"].values
temp_t_d4 = demo99_df["weld_time_s"].values
temp_dT_lo_d4 = ((Q_min_train - Q_min_err_train) / (temp_I_d4 ** 2 * temp_t_d4) - 1.0) / ALPHA_R_PER_C
temp_dT_hi_d4 = ((Q_min_train + Q_min_err_train) / (temp_I_d4 ** 2 * temp_t_d4) - 1.0) / ALPHA_R_PER_C
x_temp_zone_d4 = np.arange(n_frames_d4)
temp_ylo_d4, temp_yhi_d4 = ax_temp_d4.get_ylim()
ax_temp_d4.fill_between(x_temp_zone_d4, temp_ylo_d4, temp_dT_lo_d4, color="#f8d7d7", zorder=0.5)
ax_temp_d4.fill_between(x_temp_zone_d4, temp_dT_lo_d4, temp_dT_hi_d4, color="#ffe9a8", zorder=0.5)
ax_temp_d4.fill_between(x_temp_zone_d4, temp_dT_hi_d4, temp_yhi_d4, color="#d5f2d5", zorder=0.5)
temp_zone_legend_d4 = [Patch(facecolor="#f8d7d7", edgecolor="gray", label="Risk"),
                        Patch(facecolor="#ffe9a8", edgecolor="gray", label="Caution"),
                        Patch(facecolor="#d5f2d5", edgecolor="gray", label="Safe")]
temp_zone_legend_artist_d4 = ax_temp_d4.legend(handles=temp_zone_legend_d4, loc="lower right", fontsize=7, framealpha=0.9)
ax_temp_d4.add_artist(temp_zone_legend_artist_d4)

ax_temp_d4.set_xlabel("Sample Sequence"); ax_temp_d4.set_ylabel("Temp Deviation (\u00b0C)")
ax_temp_d4.set_title("Surface Temp Deviation:\n SIMULATED Truth vs Sensor Obs", fontweight="bold")
ax_temp_d4.grid(True, ls=":", alpha=0.4)
ax_temp_d4.axhline(0, color="black", ls="--", lw=1, zorder=2)
temp_true_line, = ax_temp_d4.plot([], [], color="#d62728", lw=2, label="SIMULATED Truth")
temp_obs_line, = ax_temp_d4.plot([], [], color="#1f77b4", lw=1.5, alpha=0.7, label="Sensor Obs")
temp_true_point, = ax_temp_d4.plot([], [], marker="o", color="#d62728", markersize=7, zorder=5)
temp_obs_point, = ax_temp_d4.plot([], [], marker="o", color="#1f77b4", markersize=7, zorder=5)
ax_temp_d4.legend(loc="upper left")

# Row 2 Col 2 -- Tilt Deviation
curv_pad = 2.0
ax_curv_d4.set_xlim(0, max(n_frames_d4 - 1, 1))
ax_curv_d4.set_ylim(tilt_lo_d4, tilt_hi_d4)  # Match y-axis limits with Row 2 Col 3

# Row 2 Col 2 Background: Calculate required tilt deviation bounds for effective angle limits
curv_angle_d4 = demo99_df["angle_deg"].values
curv_dtilt_lo_d4 = (EXPLODE_TILT_THRESHOLD_DEG - TILT_BAND_DEG) - curv_angle_d4  # Safe if smaller
curv_dtilt_hi_d4 = (EXPLODE_TILT_THRESHOLD_DEG + TILT_BAND_DEG) - curv_angle_d4  # Risk if larger
x_curv_zone_d4 = np.arange(n_frames_d4)
curv_ylo_d4, curv_yhi_d4 = ax_curv_d4.get_ylim()
ax_curv_d4.fill_between(x_curv_zone_d4, curv_ylo_d4, curv_dtilt_lo_d4, color="#d5f2d5", zorder=0.5)
ax_curv_d4.fill_between(x_curv_zone_d4, curv_dtilt_lo_d4, curv_dtilt_hi_d4, color="#ffe9a8", zorder=0.5)
ax_curv_d4.fill_between(x_curv_zone_d4, curv_dtilt_hi_d4, curv_yhi_d4, color="#f8d7d7", zorder=0.5)
curv_zone_legend_d4 = [Patch(facecolor="#f8d7d7", edgecolor="gray", label="Risk"),
                        Patch(facecolor="#ffe9a8", edgecolor="gray", label="Caution"),
                        Patch(facecolor="#d5f2d5", edgecolor="gray", label="Safe")]
curv_zone_legend_artist_d4 = ax_curv_d4.legend(handles=curv_zone_legend_d4, loc="lower right", fontsize=7, framealpha=0.9)
ax_curv_d4.add_artist(curv_zone_legend_artist_d4)

ax_curv_d4.set_xlabel("Sample Sequence"); ax_curv_d4.set_ylabel("Tilt Deviation (deg)")
ax_curv_d4.set_title("Tilt Deviation:\n SIMULATED Truth vs Sensor Obs", fontweight="bold")
ax_curv_d4.grid(True, ls=":", alpha=0.4)
ax_curv_d4.axhline(0, color="black", ls="--", lw=1, zorder=2)
curv_true_line, = ax_curv_d4.plot([], [], color="#d62728", lw=2, label="SIMULATED Truth")
curv_obs_line, = ax_curv_d4.plot([], [], color="#1f77b4", lw=1.5, alpha=0.7, label="Sensor Obs")
curv_true_point, = ax_curv_d4.plot([], [], marker="o", color="#d62728", markersize=7, zorder=5)
curv_obs_point, = ax_curv_d4.plot([], [], marker="o", color="#1f77b4", markersize=7, zorder=5)
ax_curv_d4.legend(loc="upper left")

# Row 3 Col 3 -- Bad Probability Fitting Curve
ax_curve_d4.errorbar(bad_binned_d4["heat_mean"], bad_binned_d4["bad_rate"], yerr=bad_binned_d4["bad_rate_sem"],
                      fmt="o", color="#1f77b4", ecolor="#d62728", capsize=4, markersize=6, zorder=3,
                      label="Empirical Interval (Mean ± SEM)")
ax_curve_d4.plot(Q_curve_d4, bad_curve_d4, color="#2ca02c", lw=2, label="Fitted Bad Prob Curve")
ax_curve_d4.axhline(0.5, color="gray", ls=":", lw=1)
ax_curve_d4.set_xlabel("Heat Input $Q$"); ax_curve_d4.set_ylabel("Bad Probability (Fitted)")
ax_curve_d4.set_ylim(-0.05, 1.05)
ax_curve_d4.set_xlim(*Q_XLIM_D4)
ax_curve_d4.set_title("Bad Prob Fitting Function:\nSIMULATED Truth vs baseline vs adaptive", fontweight="bold")
ax_curve_d4.grid(True, ls=":", alpha=0.4)
true_curve_point, = ax_curve_d4.plot([], [], marker="*", markersize=20, color="black",
                                       markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="SIMULATED Truth")
base_curve_point, = ax_curve_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                       markeredgecolor="black", zorder=5, label="baseline")
adapt_curve_point, = ax_curve_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                        markeredgecolor="black", zorder=5, label="adaptive")
ax_curve_d4.legend(loc="upper right", fontsize=9)

# Row 3 Col 2 -- Nugget Diameter Growth Curve (Reuse D-2 fit)
ax_nugget_d4.errorbar(bad_binned_d4["heat_mean"], bad_binned_d4["nugget_mean"], yerr=bad_binned_d4["nugget_sem"],
                       fmt="o", color="#1f77b4", ecolor="#d62728", capsize=4, markersize=6, zorder=3,
                       label="Empirical Interval (Mean ± SEM)")
ax_nugget_d4.plot(Q_curve_d4, nugget_curve_d4, color="#9467bd", lw=2, label="Fitted Nugget Growth Curve")
ax_nugget_d4.set_xlabel("Heat Input $Q$"); ax_nugget_d4.set_ylabel("Predicted Nugget Dia. (mm)")
ax_nugget_d4.set_xlim(*Q_XLIM_D4)
ax_nugget_d4.set_title("Nugget Dia. Growth Curve:\nSIMULATED Truth vs baseline vs adaptive", fontweight="bold")
ax_nugget_d4.grid(True, ls=":", alpha=0.4)
true_nugget_point, = ax_nugget_d4.plot([], [], marker="*", markersize=20, color="black",
                                         markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="SIMULATED Truth")
base_nugget_point, = ax_nugget_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                         markeredgecolor="black", zorder=5, label="baseline")
adapt_nugget_point, = ax_nugget_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                          markeredgecolor="black", zorder=5, label="adaptive")
ax_nugget_d4.legend(loc="lower right", fontsize=9)

# Row 3 Col 1 -- Tensile Strength Prediction (Nugget-Tensile linear regression)
ax_pull_d4.errorbar(pull_binned_d4["nugget_mean"], pull_binned_d4["pull_mean"],
                     xerr=pull_binned_d4["nugget_sem"], yerr=pull_binned_d4["pull_sem"],
                     fmt="o", color="#1f77b4", ecolor="#d62728", capsize=4, markersize=6, zorder=3,
                     label="Empirical Interval (Mean ± SEM)")
ax_pull_d4.plot(nugget_range_d4, pull_curve_d4, color="#ff7f0e", lw=2,
                 label=f"Linear Fit (r={pull_lr_d4.rvalue:.2f})")
ax_pull_d4.set_xlabel("Predicted Nugget Dia. (mm)"); ax_pull_d4.set_ylabel("Predicted Tensile Strength (N)")
ax_pull_d4.set_title("Tensile Strength Prediction:\nSIMULATED Truth vs baseline vs adaptive", fontweight="bold")
ax_pull_d4.grid(True, ls=":", alpha=0.4)
true_pull_point, = ax_pull_d4.plot([], [], marker="*", markersize=20, color="black",
                                     markeredgecolor="gold", markeredgewidth=1.5, zorder=6, label="SIMULATED Truth")
base_pull_point, = ax_pull_d4.plot([], [], marker="o", markersize=11, color="#7f7f7f",
                                     markeredgecolor="black", zorder=5, label="baseline")
adapt_pull_point, = ax_pull_d4.plot([], [], marker="o", markersize=11, color="#2ca02c",
                                      markeredgecolor="black", zorder=5, label="adaptive")
ax_pull_d4.legend(loc="lower right", fontsize=9)

# Row 1 Col 3 -- Process Phase Diagram (Current x Time): 
# Same definition as heat_input_proxy (assuming R=1)
PHASE_I_MAX_D4 = 5000.0
PHASE_T_MAX_D4 = 2.0
I_grid_d4 = np.linspace(0.0, PHASE_I_MAX_D4, 220)
t_grid_d4 = np.linspace(0.0, PHASE_T_MAX_D4, 220)
II_real_d4, TT_time_d4 = np.meshgrid(I_grid_d4, t_grid_d4)
QQ_real_d4 = II_real_d4 ** 2 * TT_time_d4  
zone_real_d4 = np.where(QQ_real_d4 < Q_min_train - Q_min_err_train, 0,
                         np.where(QQ_real_d4 < Q_min_train + Q_min_err_train, 1, 2))
cat_colors_d4 = {'Good': '#2ca02c', 'Bad': '#d62728', 'Explode': '#ff7f0e'}


def draw_realphase_bg_d4(ax, angle):
    ax.clear()
    ax.contourf(II_real_d4, TT_time_d4, zone_real_d4, levels=[-0.5, 0.5, 1.5, 2.5], cmap=zone_cmap_d4)
    cs = ax.contour(II_real_d4, TT_time_d4, QQ_real_d4, levels=8, colors='gray', linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=8, fmt='%.0e')
    # Indicate position with dashed lines as the boundary is narrow
    ax.contour(II_real_d4, TT_time_d4, QQ_real_d4, levels=[Q_min_train - Q_min_err_train, Q_min_train + Q_min_err_train], colors='#b8860b', linestyles='--', linewidths=1.2)  
    for cat, color in cat_colors_d4.items():
        s = agg[(agg["category"] == cat) & (agg["angle_deg"] == angle)]
        ax.scatter(s["avg_current_A"], s["weld_time_s"], s=12, color=color, alpha=0.3)
    ax.set_xlabel("Current (A)"); ax.set_ylabel("Weld Time (s)")
    ax.set_xlim(0, PHASE_I_MAX_D4); ax.set_ylim(0, PHASE_T_MAX_D4)
    ax.grid(True, ls=":", color="gray", alpha=0.5, zorder=1.5)
    zone_legend_real_d4 = [Patch(facecolor='#f8d7d7', edgecolor='gray', label='Risk'),
                            Patch(facecolor='#ffe9a8', edgecolor='gray', label='Caution'),
                            Patch(facecolor='#d5f2d5', edgecolor='gray', label='Safe')]
    ax.legend(handles=zone_legend_real_d4, loc='upper right', fontsize=8, framealpha=0.9)

fig.suptitle("RSW Adaptive Fine-Tuning Real-Time Dashboard (99 Empirical IR Samples)",
             fontsize=17, fontweight="bold")


def update_d4(i):
    row = demo99_df.iloc[i]
    im_artist_d4.set_data(frames_d4[i])
    label_text_d4.set_text(f"Sample #{int(row['sample_id'])}")
    angle_text_d4.set_text(f"Electrode Angle {row['angle_deg']:g}\u00b0")

    baseline_line_d4.set_data(np.arange(i + 1), baseline_rolling_acc_d4[:i + 1])
    adaptive_line_d4.set_data(np.arange(i + 1), adaptive_rolling_acc_d4[:i + 1])
    baseline_point_d4.set_data([i], [baseline_rolling_acc_d4[i]])
    adaptive_point_d4.set_data([i], [adaptive_rolling_acc_d4[i]])

    true_point_d4.set_data([Q_true_arr[i]], [tilt_true_arr[i]])
    base_point_d4.set_data([Q_base_arr[i]], [tilt_base_arr[i]])
    adapt_point_d4.set_data([Q_adapt_arr[i]], [tilt_adapt_arr[i]])

    temp_true_line.set_data(np.arange(i + 1), temp_true_arr[:i + 1])
    temp_obs_line.set_data(np.arange(i + 1), temp_obs_arr[:i + 1])
    temp_true_point.set_data([i], [temp_true_arr[i]])
    temp_obs_point.set_data([i], [temp_obs_arr[i]])

    curv_true_line.set_data(np.arange(i + 1), curv_true_arr[:i + 1])
    curv_obs_line.set_data(np.arange(i + 1), curv_obs_arr[:i + 1])
    curv_true_point.set_data([i], [curv_true_arr[i]])
    curv_obs_point.set_data([i], [curv_obs_arr[i]])

    true_curve_point.set_data([Q_true_arr[i]], [bad_rate_logistic_model(Q_true_arr[i], Q_min_train, k_train)])
    base_curve_point.set_data([Q_base_arr[i]], [bad_rate_logistic_model(Q_base_arr[i], Q_min_train, k_train)])
    adapt_curve_point.set_data([Q_adapt_arr[i]], [bad_rate_logistic_model(Q_adapt_arr[i], Q_min_train, k_train)])

    true_nugget_point.set_data([Q_true_arr[i]], [nugget_true_arr[i]])
    base_nugget_point.set_data([Q_base_arr[i]], [nugget_base_arr[i]])
    adapt_nugget_point.set_data([Q_adapt_arr[i]], [nugget_adapt_arr[i]])

    true_pull_point.set_data([nugget_true_arr[i]], [pull_true_arr[i]])
    base_pull_point.set_data([nugget_base_arr[i]], [pull_base_arr[i]])
    adapt_pull_point.set_data([nugget_adapt_arr[i]], [pull_adapt_arr[i]])

    # Row 1 Col 3 (redraw points/text as draw_realphase_bg_d4 clears ax)
    draw_realphase_bg_d4(ax_realphase_d4, row["angle_deg"])
    ax_realphase_d4.scatter([row["avg_current_A"]], [row["weld_time_s"]], s=400, marker="*",
                             color="black", edgecolor="yellow", linewidth=1.5, zorder=5)
    
    # Based on effective contact angle including tilt deviation
    explode_tag_d4 = "Safe" if row["effective_tilt_true_deg"] <= EXPLODE_TILT_THRESHOLD_DEG else "Risk \u2191"  
    ax_realphase_d4.set_title(f"Phase Diagram (Empirical): \nAngle {row['angle_deg']:g}\u00b0 (Explode {explode_tag_d4})",
                               fontweight="bold")
    ax_realphase_d4.text(
        0.98, 0.02,
        f"Current {row['avg_current_A']:.0f}A / Time {row['weld_time_s']:.2f}s\n"
        f"Pressure {row['pressure_psi']:.0f}PSI / Angle {row['angle_deg']:g}\u00b0",
        transform=ax_realphase_d4.transAxes, fontsize=10, fontweight="bold", va="bottom", ha="right",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, pad=0.5))

    return (im_artist_d4, label_text_d4, angle_text_d4, baseline_line_d4, adaptive_line_d4,
            baseline_point_d4, adaptive_point_d4, true_point_d4, base_point_d4, adapt_point_d4,
            temp_true_line, temp_obs_line, temp_true_point, temp_obs_point,
            curv_true_line, curv_obs_line, curv_true_point, curv_obs_point,
            true_curve_point, base_curve_point, adapt_curve_point,
            true_nugget_point, base_nugget_point, adapt_nugget_point,
            true_pull_point, base_pull_point, adapt_pull_point)


ANIM_INTERVAL_MS_D4 = 250
ANIM_FPS_D4 = 4
anim_d4 = FuncAnimation(fig, update_d4, frames=n_frames_d4, interval=ANIM_INTERVAL_MS_D4, blit=False)

gif_path_d4 = os.path.join(FIGURE_DIR, "adaptive_dashboard_eng.gif")
anim_d4.save(gif_path_d4, writer=PillowWriter(fps=ANIM_FPS_D4))
print(f"[Done] Saved {n_frames_d4}-frame 3x3 dashboard (GIF): {gif_path_d4}")

plt.rcParams["animation.embed_limit"] = 80
from IPython.display import HTML, display as ipy_display
interactive_player_d4 = HTML(anim_d4.to_jshtml())
plt.close(fig)
ipy_display(interactive_player_d4)
print(f"[Final] baseline cumulative accuracy {baseline_rolling_acc_d4[-1] * 100:.1f}%, "
      f"adaptive cumulative accuracy {adaptive_rolling_acc_d4[-1] * 100:.1f}%")